# Run FBA simulations for the individual oryzae isolates

In [1]:
import medusa
import cobra
from cobra.flux_analysis.loopless import loopless_solution
import pandas as pd
import numpy as np
from pathlib import Path
from cobra.io import read_sbml_model
from medusa.flux_analysis import flux_balance
from copy import deepcopy
from pickle import load
from cobra.flux_analysis import gapfill

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

# Importing helper functions
from helpers import reactionToComp, geneToComp, add

## BioLog simulations

Here, we will perform FBA simulations according to the experiments conducted by Casper. For eight isolates, metabolic activity on different carbon and nitrogen sources was experimentally determined. Here, we will emulate the different media in our simulations. Ideally, our model predictions should coincide with the experimentally observed metabolic activities from the BioLog phenotype microarray experiment conducted by Casper

### Load model

In [2]:
# Load the ensemble model from constructEnsemble.ipynb
with open("./scrap/ensemble.pickle", 'rb') as infile:
    ensemble_save = load(infile)

# deepcopy to have an object to fall back to
ensemble = deepcopy(ensemble_save)

Set parameter Username
Academic license - for non-commercial use only - expires 2025-09-26
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpuw7qqmom.lp
Reading time = 0.01 seconds
: 2152 rows, 4500 columns, 17072 nonzeros
Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmpuvvxtjn8.lp
Reading time = 0.01 seconds
: 2152 rows, 4500 columns, 17072 nonzeros


## Load BPGA gene table

To assign genes to gapfilled reactions if possible.

In [3]:
# Import BPGA table 
import pandas as pd
BPGA = pd.read_csv('BPGA2ortho_GEM_custom.csv', delimiter=";", dtype=str)

### Specify the target metabolites

In [4]:
conditions = pd.read_csv('../BioLog/metaboliteDf.csv', delimiter=";", dtype=str)
conditions = conditions.sort_values(by=['basePlus150', 'basePlus100'], ascending=[False, False])
conditions.head(40)

,Plate,Well,Medium,Condition,basePlus100,basePlus150,KEGG
1,PM01,A02,L_Arabinose,L_Arabinose_PM01,8,8,C00259[e]
19,PM01,B08,D_Xylose,D_Xylose_PM01,8,8,C00181[e]
27,PM01,C04,D_Ribose,D_Ribose_PM01,8,8,C00121[e]
89,PM01,H06,L_Lyxose,L_Lyxose_PM01,8,8,NaN
115,PM02A,B08,Arbutin,Arbutin_PM02A,8,8,NaN
276,PM03B,H01,Ala_Asp,Ala_Asp_PM03B,8,8,NaN
215,PM03B,B12,L_Tryptophan,L_Tryptophan_PM03B,8,7,C00078[e]
287,PM03B,H12,Met_Ala,Met_Ala_PM03B,8,7,NaN
188,PM02A,H09,Dihydroxy_Acetone,Dihydroxy_Acetone_PM02A,8,6,C00184[c]
29,PM01,C06,L_Rhamnose,L_Rhamnose_PM01,7,6,NaN


### Specify the medium

In [5]:
# Setup medium definitions
ensemble.base_model.reactions.get_by_id("r2205").bounds = (0,1000.0) # in stead of 1,1 or 1,1000, otherwise we get 1,0 -> invalid
ensemble.base_model.medium

{'r2199': 1000.0,
 'r2200': 1000.0,
 'r2202': 1000.0,
 'r2203': 1000.0,
 'r2204': 1000.0,
 'r2205': 1000.0}

In [6]:
# Note that model.medium is just a copy of the current exchange fluxes. 
# Assigning to it directly with model.medium[...] = ... will not change the model. 
# You have to assign an entire dictionary with the changed import flux upper bounds.

## Setup medium definition for the carbon-limiting plates
## -> assume this is media with NH3, H2SO3, and H3PO4 present, but no C-source
medium_Climit = deepcopy(ensemble.base_model.medium)
# In theory, I think we also need to set r2200 (NH3) and r2204 (H2SO3, sulfite) to zero
# However, if I do this, it always predicts no growth
medium_Climit['r2199'] = 0 # set CO2 to zero
# medium_Climit['r2200'] = 0 # do not set ammonia (NH3) to zero
# medium_Climit['r2202'] = 0 # do not set oxygen (O2) to zero
# medium_Climit['r2203'] = 0 # do not set phosphate (H2SO3) to zero
# medium_Climit['r2204'] = 0 # do not set sulifte (H2PO3) to zero
medium_Climit['r2205'] = 0 # set glucose to zero
medium_Climit

## Setup medium definition for the nitrogen-limiting plates
## -> assume this is media with Glucose, H2SO3, and H3PO4 present, but no N-source
medium_Nlimit = deepcopy(ensemble.base_model.medium)
medium_Nlimit['r2199'] = 0 # set CO2 to zero
medium_Nlimit['r2200'] = 0 # set ammonia (NH3) to zero
medium_Nlimit

{'r2199': 0,
 'r2200': 1000.0,
 'r2202': 1000.0,
 'r2203': 1000.0,
 'r2204': 1000.0,
 'r2205': 0}

{'r2199': 0,
 'r2200': 0,
 'r2202': 1000.0,
 'r2203': 1000.0,
 'r2204': 1000.0,
 'r2205': 1000.0}

### Subset to relevant oryzae isolates

In [7]:
submembers = ['Pan_oryzae', 'template', 'Aspergillus_oryzae_RIB40_CAoGD', 'Aspergillus_oryzae_NRRL_2217','Aspergillus_oryzae_NRRL_3483',
              'Aspergillus_oryzae_NRRL_5589', 'Aspergillus_oryzae_NRRL_3488', 'Aspergillus_oryzae_NRRL_5592',
              'Aspergillus_oryzae_NRRL_35890', 'Aspergillus_oryzae_NRRL_471'] # take all eight, would be much nicer if done with regexp

### Curate model

#### Fix reaction r205

By doing this, our simulation no longer predict growth on media without P, but still predict growth for media with P. 

In [8]:
# Add metabolite myo-Inositol 4-phosphate to the model
C03546 = cobra.Metabolite(id="C03546[c]", compartment="c", name="myo-Inositol 4-phosphate")
ensemble.base_model.add_metabolites([C03546])

# Fix reaction r205
r205 = ensemble.base_model.reactions.get_by_id("r205")
r205.reaction = "C00001[c] + C01220[c] --> C00009[c] + C03546[c]"
ensemble.base_model.reactions.get_by_id("r205")

Reaction identifier,r205
Name,"Inositol-1,4-bisphosphate 1-phosphatase"
Memory address,0x23fcc6168d0
Stoichiometry,"C00001[c] + C01220[c] --> C00009[c] + C03546[c] H2O + 1D-myo-inositol 1,4-bisphosphate --> phosphate + myo-Inositol 4-phosphate"
GPR,
Lower bound,0
Upper bound,1000.0


### Remove reaction r191

In [9]:
# Erroneous reaction, remove it
ensemble.base_model.remove_reactions(reactions = ['r191'],
                                     remove_orphans = True)

c:\Users\gilis\OneDrive - Chalmers\Desktop\postdoc\Aspergillus\medusa\.venv\Lib\site-packages\cobra\core\group.py:147: UserWarning: need to pass in a list
  warn("need to pass in a list")


#### Fix reactions r246 and r247

In [10]:
# Fix bounds of these reactions
ensemble.base_model.reactions.get_by_id("r246").bounds = (0,1000)
ensemble.base_model.reactions.get_by_id("r247").bounds = (0,1000)

#### Fix reaction OtherAsp_R10479

- Remove that new verscolorin reaction
- Remove that new averantin reaction
- Maybe: Add the genes of the new averantin reaction in the GPR of the old averantin reaction

By doing this, we no longer predict anaerobic growth, but still allow for aerobic growth, except (!) for NRRL_35890.

In [11]:
# # Aerobic -> growth, except for NRRL_35890. Note that the pan-model chooses to grow without taking up O2
# fluxes = flux_balance.optimize_ensemble(ensemble,
#                                         specific_models = submembers,
#                                         return_flux = ['r2359', 'r2202'])
# fluxes

# # Anaerobic -> growth, except for NRRL_35890, the template model, and RIB40.
# # Notice that neither the template model nor RIB40 has gene cluster_3696, which catalyzes reaction OtherAsp_R10479
# medium_anaerobic = deepcopy(ensemble.base_model.medium)
# medium_anaerobic['r2202'] = 0
# ensemble.base_model.medium = medium_anaerobic
# fluxes = flux_balance.optimize_ensemble(ensemble,
#                                         specific_models = submembers,
#                                         return_flux = ['r2359', 'r2202'])
# fluxes

In [12]:
# Remove the new verscolorin reaction
# Remove the new averantin reaction
# - Add the genes of the new averantin reaction in the GPR of the old averantin reaction
ensemble.base_model.remove_reactions(reactions = ['OtherAsp_R10479', 'OtherAsp_R10317', 'OtherAsp_R10318'],
                                     remove_orphans = True)

# TODO: we could add the genes of the new averantin reaction in the GPR of the old averantin reaction.
# However, while all oryzae isolates should probably be capable of this reaction, the template and RIB40
# do not have these genes ('cluster_3696', 'cluster_8046', 'cluster_8086'). As such, it might be better
# to keep it gene-less...


In [13]:
# # Aerobic -> growth, except for NRRL_35890. Note that the pan-model chooses to grow without taking up O2
# medium_aerobic = deepcopy(medium_anaerobic)
# medium_aerobic['r2202'] = 1000
# ensemble.base_model.medium = medium_aerobic
# ensemble.base_model.medium
# fluxes = flux_balance.optimize_ensemble(ensemble,
#                                         specific_models = submembers,
#                                         return_flux = ['r2359', 'r2202'])
# fluxes

# # Anaerobic -> growth, except for NRRL_35890, the template model, and RIB40.
# # Notice that neither the template model nor RIB40 has gene cluster_3696, which catalyzes reaction OtherAsp_R10479
# ensemble.base_model.medium = medium_anaerobic
# ensemble.base_model.medium
# fluxes = flux_balance.optimize_ensemble(ensemble,
#                                         specific_models = submembers,
#                                         return_flux = ['r2359', 'r2202'])
# fluxes

#### Fix reaction r2095

In [14]:
# Ammonium transport across the cell membrane should be bidirectional
ensemble.base_model.reactions.get_by_id("r2095").bounds = (-1000,1000)

#### Add exchanges

In [15]:
# CO2 must be able to move out
ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id("C00011[e]"), type="demand")

# NH3 must be able to move out
ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id("C00014[e]"), type="demand")

# So now CO2 and NH3 can both be taken up, excreted, and transported to the cytoplasm

Reaction identifier,DM_C00011[e]
Name,CO2 demand
Memory address,0x23fcea52480
Stoichiometry,C00011[e] --> CO2 -->
GPR,
Lower bound,0
Upper bound,1000.0


Reaction identifier,DM_C00014[e]
Name,NH3 demand
Memory address,0x23fd17e8080
Stoichiometry,C00014[e] --> NH3 -->
GPR,
Lower bound,0
Upper bound,1000.0


#### Remove exchanges

In [16]:
ensemble.base_model.remove_reactions(reactions = ["r2281","r2282","r2283","r2284","r2285","r2286","r2287","r2288","r2289",
                                                  "r2291","r2292","r2295","r2318","r2319","r2320","r2321","r2322","r2323"])

### Add requirement for cofactors to the biomass function


Notes from meeting:

1. Add exchange so that => cofactor(c)
2. Check if there is growth if the metabolite would be intracellular of if transport reaction would be added


In [17]:
# Optional: aim to improve the biomass function by incorporating cofactors

# Add an artificial metabolite "Cofactor[c]"
M_cofactor_c = cobra.Metabolite(id="Cofactor[c]", compartment="c", name="Cofactor")
ensemble.base_model.add_metabolites([M_cofactor_c])

# Include this metabolite on the left-hand side of the biomass specification
reaction = ensemble.base_model.reactions.get_by_id("r1897")
reaction.add_metabolites({
    'Cofactor[c]': -1
})

# Specify the reaction generating Cofactor[c]
reaction = cobra.Reaction('Artificial_Cofactor_formation')
reaction.name = 'Cofactor formation'
reaction.lower_bound = 0  # This is the default
reaction.upper_bound = 1000  # This is the default
ensemble.base_model.add_reactions([reaction])

# For yeast8.7.1: 0.00019 coenzyme A[c] + 1e-05 FAD[c] + 0.00265 NAD[c] + 0.00015 NADH[c] + 0.00057 NADP(+)[c] + 0.0027 NADPH[c] + 0.00099 riboflavin[c] + 
# 1.2e-06 TDP[c] + 6.34e-05 THF[c] + 1e-06 heme a[c] => cofactor[c]

# Removing metabolites not described in Aspergillus, leaving coefficients unchanged:
# 0.00019 coenzyme A[c] + 1e-05 FAD[c] + 0.00265 NAD[c] + 0.00015 NADH[c] + 0.00057 NADP(+)[c] + 0.0027 NADPH[c] + 0.00099 riboflavin[c] + 1e-06 heme a[c] => cofactor[c]

# In addition, I am removing cytoplasmic heme a, since it is only present in 1 reaction that is entirelty disconnected from the rest of metabolism
# 'C15670[c]': -1e-06, # heme

reaction.add_metabolites({
    'C00010[c]': -0.00019, # coenzyme A[c]
    'C00016[c]': -1e-05, # FAD[c]
    'C00003[c]': -0.00265, # NAD+
    'C00004[c]': -0.00015, # NADH
    'C00006[c]': -0.00057, # NADP+
    'C00005[c]': -0.0027, # NADPH
    'C00255[c]': -0.00099, # riboflavin
    'Cofactor[c]': 1 # cofactor
})

### Gapfill NRRL_35890

This isolate fails to grow on almost everything. More specifically, it can only grow on medium with glucose and L-Histidine. There likely is a crucial reaction missing from its model. We here try to gapfil it from the pan-oryzae model.

Using gapfilling, we discovered that reaction r766 is missing specifically from isolate NRRL_35890. This reaction seems to generate a crucial precursor for purine metabolism.
It also is connected to histidine metabolism, so perhaps that explains why growth can be recovered by adding histidine! Hence, we will give this isolate access to this reaction.

In [18]:
ensemble.features.get_by_id("r766_upper_bound").states.update({'Aspergillus_oryzae_NRRL_35890': 1000})

In [19]:
# Improves prediction by allowing growth on
reaction_r692_c = deepcopy(ensemble.base_model.reactions.get_by_id("r692")) # deepcopy required
reaction_r692_c.reaction = "C00026[c] + C15767[c] --> C00025[c] + C00232[c]"
reaction_r692_c.id = 'r692_cytoplasmic'
ensemble.base_model.add_reactions([reaction_r692_c])
ensemble.base_model.reactions.get_by_id('r692_cytoplasmic')

Read LP format model from file C:\Users\gilis\AppData\Local\Temp\tmp32obrux9.lp
Reading time = 0.02 seconds
: 2153 rows, 4462 columns, 17014 nonzeros


Reaction identifier,r692_cytoplasmic
Name,4-aminobutyrate aminotransferase
Memory address,0x23fcf64ffb0
Stoichiometry,C00026[c] + C15767[c] --> C00025[c] + C00232[c] 2-oxoglutarate + gamma-aminobutyrate --> L-glutamate + succinate semialdehyde
GPR,cluster_8165
Lower bound,0
Upper bound,1000.0


## Growth on amygdalin

Only reaction related to amygdalin in GEM, + a few downstream reactions:
 - OtherAsp_R02985, amygdalin beta-glucosidase, D-glucose[c] + Prunasin[c] <=> H2O[c] + Amygdalin[c], cluster_1047
 - OtherAsp_R02558, (R)-Prunasin beta-D-glucohydrolase, H2O[c] + Prunasin[c] <=> D-glucose[c] + Mandelonitrile[c], cluster_1047
But then we get stuck, we only then have
 - OtherAsp_R10035, beta-D-glucoside glucohydrolase, H2O[c] + Dhurrin[c] <=> beta-D-glucose[c] + (S)-4-Hydroxymandelonitrile[c], cluster_1047
But Dhurrin is also a dead end.

However, in the niger GEM we have:
 - Water (extracellular)[e] + (R)-amygdalin[e] => D-Glucose (extracellular)[e] + (R)-prunasin[e]
 - Water (extracellular)[e] + (R)-prunasin[e] => D-Glucose (extracellular)[e] + (R)-Mandelonitrile[e]
 - (R)-Mandelonitrile[e] <=> (R)-Mandelonitrile[c]
 - 2 Water[c] + (R)-Mandelonitrile[c] => (R)-Mandelate[c] + Ammonia[c]
 - (R)-Mandelate[c] + NAD+[c] <=> Benzoylformate[c] + NADH[c]
 - Benzoylformate[c] => Benzaldehyde[c] + Carbon dioxide[c]
 - Benzaldehyde[c] + Water[c] + NAD+[c] <=> Benzoic acid[c] + NADH[c]
 - Benzoic acid[c] + NADPH[c] + Oxygen[c] => 4-Hydroxybenzoic acid[c] + Water[c] + NADP+[c]
 - ... more ...

It also has demands for several of these compounds, like:
 - (R)-amygdalin[e] =>
 - (R)-prunasin[e] =>
 - (R)-Mandelonitrile[e] =>

That second reaction would immediately allow growth on amygdalin, by growth on glucose and removing prunasin.

Note that the gene cluster_1047 being in the cytoplasm is a prediction by deeploc. In fact, this prediction is a bit artificial: it actually predicted it to be a transmembrane protein of the lysosome/vacuole. Since our GEM doesnt have a lysosome/vacuole, we placed it into the cytoplasm. However, the high score for transmembrane protein makes it reasonable to assume this could be a protein located on the cytoplasm/extracellular membrane.

Based on the niger GEM and based on some literature, it is possible that the pathway that is described in niger is also present in oryzae. However, none of the intermediate metabolites (Mandelate, Benzoylformate, Benzaldehyde) exist in the GEM. Hence, it is more likely that this pathway isnt there. As such, I will just add the first two steps that are catalyzed by gene cluster_1047, and then assume Mandelonitrile is excreted.
Below, I add this pathway.


In [20]:
# Step 1: turn on the gene cluster_1047 in all isolates of interest for this reaction
for model, bound in ensemble.features.OtherAsp_R02985_lower_bound.states.items():
    if(model in submembers):
        ensemble.features.OtherAsp_R02985_lower_bound.states.update({model: -1000})

# Step 2: Displace the reaction to break down amygdalin to the extracellular space. This requires putting the metabolites into the extracellular space first
# Step 2.1: Add metabolites in the extracellular space
C08325 = cobra.Metabolite(id="C08325[e]", compartment="e", name="Amygdalin")
ensemble.base_model.add_metabolites([C08325])
C00844 = cobra.Metabolite(id="C00844[e]", compartment="e", name="Prunasin")
ensemble.base_model.add_metabolites([C00844])

# Step 2.2: Move reaction OtherAsp_R02985 to the extracellular space
OtherAsp_R02985 = ensemble.base_model.reactions.get_by_id("OtherAsp_R02985")
OtherAsp_R02985.reaction = "C00031[e] + C00844[e] --> C00001[e] + C08325[e]"

# Step 3: turn on the gene cluster_1047 in all isolates of interest for this reaction
for model, bound in ensemble.features.OtherAsp_R02558_upper_bound.states.items():
    if(model in submembers):
        ensemble.features.OtherAsp_R02558_upper_bound.states.update({model: 1000})

# Step 4: Displace the reaction to break down Prunasin to the extracellular space. This requires putting the metabolites into the extracellular space first
# Step 4.1: Add metabolite in the extracellular space
C00561 = cobra.Metabolite(id="C00561[e]", compartment="e", name="Mandelonitrile")
ensemble.base_model.add_metabolites([C00561])

# Step 4.2: Move reaction OtherAsp_R02985 to the extracellular space
OtherAsp_R02558 = ensemble.base_model.reactions.get_by_id("OtherAsp_R02558")
OtherAsp_R02558.reaction = "C00001[e] + C00844[e] --> C00031[e] + C00561[e]"

# Step 5: Add reaction for excreting Mandelonitrile[e]
ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id("C00561[e]"), type="demand")

Reaction identifier,DM_C00561[e]
Name,Mandelonitrile demand
Memory address,0x23fdf4db110
Stoichiometry,C00561[e] --> Mandelonitrile -->
GPR,
Lower bound,0
Upper bound,1000.0


## Growth on stachyose

Only 1 reaction:

cluster_7612, H2O[c] + Stachyose[c] <=> D-fructose[c] + D-Gal alpha 1->6D-Gal alpha 1->6D-Glucose[c]

No growth predicted obviously, because (1) stachyose cant get in and (2) D-Gal alpha 1->6D-Gal alpha 1->6D-Glucose[c] only exists in this reaction

TODO: move the reaction (actually, the gene, so all reactions catalyzed by this gene) to the extracellular space and allow for D-Gal alpha 1->6D-Gal alpha 1->6D-Glucose[c] excretion.
Also, rename this to manninotriose.

After this, isolates that express cluster_7612 are able to grow on stachyose.

In [21]:
ensemble.base_model = geneToComp(model = ensemble.base_model, gene_id = 'cluster_7612', new_compartment = 'e')

Created new metabolite: C01613[e]
Created new metabolite: C05404[e]
Created new metabolite: C16688[e]
Created new metabolite: C00668[e]


In [22]:
ensemble.base_model.metabolites.get_by_id("C05404[e]").name = "Manninotriose"

# We have no evidence of a reaction that can further break down Manninotriose, so allow it out
new_demand = ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id("C05404[e]"), type="demand")

### Growth on dipeptides

Currently, I cannot assess growth on dipeptides. The easiest way for doing this, probably, would be to add the relevant dipeptides as novel metabolites to the GEM, allow the dipeptides to be transported inside, and subsequently add a "fake reaction" that cleaves the dipeptide into the individual amino acids.

* While it could also be that the cleaving is done extracelullarly and subsequently the amino acids are transported, this (1) does not matter for our simulations (2) there is also evidence that di- and tripeptides are transported inside directly (see https://pubmed.ncbi.nlm.nih.gov/33604648/).
* There are 136 identified proteases in A. oryzae (https://www.nature.com/articles/nature04300#MOESM1), and while it can be expected that not all enzymes are capable of cleaving all dipeptides, I do not think it is worth the effort to try to assign genes to specific reactions here.

TODO we did not yet identify genes responsible for cleaving these particular dipeptides.

In [23]:
# Add metabolite Ala_Asp to the model
Ala_Aspe = cobra.Metabolite(id="Ala_Asp[e]", compartment="e", name="Ala_Asp")
Ala_Aspc = cobra.Metabolite(id="Ala_Asp[c]", compartment="c", name="Ala_Asp")
ensemble.base_model.add_metabolites([Ala_Aspe])
ensemble.base_model.add_metabolites([Ala_Aspc])
conditions.loc[conditions['Medium'] == 'Ala_Asp', 'KEGG'] = 'Ala_Asp[e]' # also update in conditions DF

# Add a Ala_Asp transporter
reaction = cobra.Reaction('Artificial_Ala_Asp_uptake')
reaction.name = 'Ala_Asp uptake'
reaction.lower_bound = 0  # This is the default
reaction.upper_bound = 1000  
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'Ala_Asp[e]': -1,
    'Ala_Asp[c]': 1
})

# Add a reaction cleaving Ala_Asp
reaction = cobra.Reaction('Artificial_Ala_Asp_cleavage')
reaction.name = 'Ala_Asp cleavage'
reaction.lower_bound = 0  # This is the default
reaction.upper_bound = 1000  
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'Ala_Asp[c]': -1,
    'C00041[c]': 1, # Alanine
    'C00049[c]': 1, # Aspartate (Aspartic acid)
})

In [24]:
# Add metabolite Met_Ala to the model
Met_Alae = cobra.Metabolite(id="Met_Ala[e]", compartment="e", name="Met_Ala")
Met_Alac = cobra.Metabolite(id="Met_Ala[c]", compartment="c", name="Met_Ala")
ensemble.base_model.add_metabolites([Met_Alae])
ensemble.base_model.add_metabolites([Met_Alac])
conditions.loc[conditions['Medium'] == 'Met_Ala', 'KEGG'] = 'Met_Ala[e]' # also update in conditions DF

# Add a Met_Ala transporter
reaction = cobra.Reaction('Artificial_Met_Ala_uptake')
reaction.name = 'Met_Ala uptake'
reaction.lower_bound = 0  # This is the default
reaction.upper_bound = 1000  
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'Met_Ala[e]': -1,
    'Met_Ala[c]': 1
})

# Add a reaction cleaving Met_Ala
reaction = cobra.Reaction('Artificial_Met_Ala_cleavage')
reaction.name = 'Met_Ala cleavage'
reaction.lower_bound = 0  # This is the default
reaction.upper_bound = 1000  
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'Met_Ala[c]': -1,
    'C00041[c]': 1, # Alanine
    'C00073[c]': 1, # Methionine
})

#### Growth on Tyramine

Growth on Tyramine would require (1) a Tyramine transporter and (2) gapfilling of the reaction that converts 4-Hydroxyphenylacetaldehyde into something
Aspergillus oryzae can further metabolize. There are two options for this:
1. 4-Hydroxyphenylacetaldehyde + CO2 <=> 3-(4-Hydroxyphenyl)pyruvate. This achieved by enzyme 4.1.80, which is not described in Aspergillus oryzae. 
However, the Gibbs free energy for this reaction is 69, so it is unlikely to happen in this direction.

Aleternatively, two reactions may happen:

2. 4-Hydroxyphenylacetaldehyde + NAD+ + H2O <=> 4-Hydroxyphenylacetate + NADH + H+ with Gibbs of -38
3. 4-Hydroxyphenylacetate + Oxygen + NADH <=> Homogentisate + NAD+ + H2O with Gibbs of -443

As such, I will introduce these 2 reactions, which will also require introducing the metabolite 4-Hydroxyphenylacetate.
I could also do some effort to identify the GPR. The enzymes are 1.2.1.53 and 1.14.13.18
1.2.1.53 alo seems to be fine as 1.2.1.5, and I found https://www.uniprot.org/uniprotkb/A2QIG1/entry and https://www.uniprot.org/uniprotkb/A2QMA4/entry for A. niger
=> I find barley anything for 1.14.13.18 , it seems to exist in uniprot for three obscure bacteria...

TODO: we did not yet identify genes that support the suggested gapfilled reactions.


In [25]:
# Add metabolite Tyramine[e] and 4-Hydroxyphenylacetate[c] to the model
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00483[e]", compartment="e", name="Tyramine")]) # add extracellular Tyramine
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00642[c]", compartment="c", name="4-Hydroxyphenylacetate")]) # add required intermediate

# Add a Tyramine transporter
reaction = cobra.Reaction('Artificial_Tyramine_uptake')
reaction.name = 'Tyramine uptake'
reaction.lower_bound = 0  # This is the default
reaction.upper_bound = 1000  
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00483[e]': -1,
    'C00483[c]': 1
})

# Add the reaction gap1
reaction = cobra.Reaction('Artificial_Tyramine_gap1')
reaction.name = 'Tyramine gap1'
reaction.lower_bound = 0  # This is the default
reaction.upper_bound = 1000  
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C03765[c]': -1,
    'C00001[c]': -1,
    'C00003[c]': -1,
    'C00642[c]': 1,
    'C00004[c]': 1,
})

# Add the reaction gap2
reaction = cobra.Reaction('Artificial_Tyramine_gap2')
reaction.name = 'Tyramine gap2'
reaction.lower_bound = 0  # This is the default
reaction.upper_bound = 1000  
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00642[c]': -1,
    'C00004[c]': -1,
    'C00007[c]': -1,
    'C00544[c]': 1, # homogentisate
    'C00003[c]': 1,
    'C00001[c]': 1
})

In [26]:
# Result: growth, but only in isolates with cluster_2768 or cluster_3556
# Check which isolates we want to give access to these clusters -> probably all except for NRRL 5589
# -> but NRRL 5589 already has it, so I feel reluctant to remove it, perhaps it is lacking one of the enzymes catalyzing the reactions that I added
for model, bound in ensemble.features.OtherAsp_R02382_upper_bound.states.items():
    if(model in submembers):
        ensemble.features.OtherAsp_R02382_upper_bound.states.update({model: 1000})

### Growth on L_Lyxose

The only reaction specified in KEGG related to lyxose is L-Xylulose <=> L-Lyxose.
I can confirm that our models support growth on extracellular Xylulose, and they also support metabolic activity (albeit without needing O2 there). Hence, it is quite likely that we are simply missing this conversion reaction. Note that the Gibbs of this reaction is -13.8, so it is likely reversible. Hence, I will add this reaction. We should still decide whether we add this reaction extracellularly or in the cytoplasm. But since I cant find the gene that is governing this reaction in oryzae, this is a rather arbritary choice, and this also has no effect from the modeling perspective. As metabolic was reported for all isolates, I will give all isolates access to this reaction.

According to my searches, it is more likely for this conversion to happen intracellularly, so let's do that.

In [27]:
# Add metabolite L_Lyxose to the model
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C01508[c]", compartment="c", name="L-Lyxose")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C01508[e]", compartment="e", name="L-Lyxose")])
conditions.loc[conditions['Medium'] == 'L_Lyxose', 'KEGG'] = 'C01508[e]' # also update in conditions DF

# Add a L_Lyxose transporter
reaction = cobra.Reaction('Artificial_L_Lyxose_uptake')
reaction.name = 'L_Lyxose uptake'
reaction.lower_bound = 0  # This is the default
reaction.upper_bound = 1000  
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C01508[e]': -1,
    'C01508[c]': 1
})

# TODO: I could look for this transporter gene. It is not unlikely that it Lyxose uses the same transporter as Xylulose, given the similarity. However, no gene was 
# assigned to this transport.

# Add conversion reaction
reaction = cobra.Reaction('Artificial_L_Lyxose_gap')
reaction.name = 'L_Lyxose gap'
reaction.lower_bound = -1000  # Given the Gibbs energy
reaction.upper_bound = 1000  
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C01508[c]': -1,
    'C00312[c]': 1
})


### Growth on Arbutin

The pathway by which arbutin is degraded is a bit speculative. According to literature, the most likely pathway to me seems:
1. Arbutin + H2O => Glucose + Hydroquinone (https://www.sciencedirect.com/science/article/pii/S1389172315001462?via%3Dihub, which suggests genes AO090038000223 = cluster_3238 and AO090003001511 = cluster_2538 for this)
2. Hydroquinone <=> Hydroquinone
3. Hydroquinone + CO2 <=> 2,5-Dihydroxybenzoate 
4. 2,5-Dihydroxybenzoate + Oxygen <=> Maleylpyruvate
5. Maleylpyruvate <=> 3-Fumarylpyruvate
6. 3-Fumarylpyruvate + H2O <=> Fumarate + Pyruvate

This does require introducing a few genes and metabolites to the model (gapfilling). For some of the genes, I do not find any direct support in the literature in Aspergillus oryzae.

In [28]:
# Add metabolite Arbutin and downstream pathway metabolites to the model
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C06186[e]", compartment="e", name="Arbutin")])
conditions.loc[conditions['Medium'] == 'Arbutin', 'KEGG'] = 'C06186[e]' # also update in conditions DF
ensemble.base_model.metabolites.get_by_id("C15603[e]").name = "Hydroquinone" # fix name
ensemble.base_model.metabolites.get_by_id("C15603[c]").name = "Hydroquinone" # fix name
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C02167[c]", compartment="c", name="3-Maleylpyruvate")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C02514[c]", compartment="c", name="3-Fumarylpyruvate")])

# Add required reactions
# r1
reaction = cobra.Reaction('Gap_Arbutin_r1')
reaction.name = 'Arbutin hydrolysis'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C06186[e]': -1,
    'C00001[e]': -1,
    'C00221[e]': 1,
    'C15603[e]': 1
})
ensemble.base_model.reactions.get_by_id("Gap_Arbutin_r1").gene_reaction_rule = 'cluster_3238 or cluster_2538' # safely add it to all submembers, it is a core gene

# r2
reaction = cobra.Reaction('Gap_Arbutin_r2')
reaction.name = 'Hydroquinone transport'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C15603[e]': -1,
    'C15603[c]': 1,
})

# r3
reaction = cobra.Reaction('Gap_Arbutin_r3')
reaction.name = 'Arbutin r3'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C15603[c]': -1,
    'C00011[c]': -1,
    'C00628[c]': 1,
})

# r4
reaction = cobra.Reaction('Gap_Arbutin_r4')
reaction.name = 'Arbutin r4'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00628[c]': -1,
    'C00007[c]': -1,
    'C02167[c]': 1
})
ensemble.base_model.reactions.get_by_id("Gap_Arbutin_r4").gene_reaction_rule = 'cluster_11847' # safely add it to all submembers, it is a core gene

# r5
reaction = cobra.Reaction('Gap_Arbutin_r5')
reaction.name = 'Arbutin r5'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C02167[c]': -1,
    'C02514[c]': 1
})

# r6
reaction = cobra.Reaction('Gap_Arbutin_r6')
reaction.name = 'Arbutin r6'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C02514[c]': -1,
    'C00122[c]': 1,
    'C00022[c]': 1
})
ensemble.base_model.reactions.get_by_id("Gap_Arbutin_r6").gene_reaction_rule = 'cluster_14600' # safely add it to all submembers, it is a core gene

### Growth on L-Rhamnose

The fungal pathway for catabolysis of L-Rhamnose is described in https://link.springer.com/article/10.1007/s00253-014-5607-9/figures/4 for Aspergillus niger.

The reactions involved are:
1. L-Rhamnose <=> L-Rhamnose (C00507)
2. L-Rhamnose + NADP+ <=> L-Rhamnono-1,4-lactone + NADPH (C00507 + C00006 <=> C02991 + C00005 + C00080)
3. L-Rhamnono-1,4-lactone + H2O <=> L-Rhamnonate (C02991 + C00001 <=> C01934) -> I already have this reaction, Ani_r1902 by cluster_9524
4. L-Rhamnonate <=> 2-Dehydro-3-deoxy-L-rhamnonate + H2O (C01934 <=> C03979 + C00001)
5. 2-Dehydro-3-deoxy-L-rhamnonate <=> (S)-Lactaldehyde + Pyruvate (C03979 <=> C00424 + C00022)

In [29]:
# Add metabolite L-Rhamnose and downstream pathway metabolites to the model
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00507[e]", compartment="e", name="L-Rhamnose")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00507[c]", compartment="c", name="L-Rhamnose")])
conditions.loc[conditions['Medium'] == 'L_Rhamnose', 'KEGG'] = 'C00507[e]' # also update in conditions DF
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C03979[c]", compartment="c", name="2-Dehydro-3-deoxy-L-rhamnonate")])

# Add required reactions
# r1
reaction = cobra.Reaction('Gap_Rhamnose_r1')
reaction.name = 'Rhamnose uptake'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00507[e]': -1,
    'C00507[c]': 1
})

# r2
reaction = cobra.Reaction('Gap_Rhamnose_r2')
reaction.name = 'Rhamnose r2'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00507[c]': -1,
    'C00006[c]': -1,
    'C02991[c]': 1,
    'C00005[c]': 1
})

# r3
# we already have the 3rd reaction in this pathway (Ani_1902, cluster_9524)

# r4
reaction = cobra.Reaction('Gap_Rhamnose_r4')
reaction.name = 'Rhamnose r4'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C01934[c]': -1,
    'C00001[c]': 1,
    'C03979[c]': 1,
})

# r5
reaction = cobra.Reaction('Gap_Rhamnose_r5')
reaction.name = 'Rhamnose r5'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C03979[c]': -1,
    'C00022[c]': 1,
    'C00424[c]': 1, # C00424 is S-lactaldehyde, C00937 is lactaldehyde
})

# This allows for growth, but RIB40 is a FN
# Try gapfilling: according to gafilling, Ani_r1902 would fix growth for RIB40
ensemble.features.Ani_r1902_upper_bound.states.update({'Aspergillus_oryzae_RIB40_CAoGD': 1000})

### Growth on 4-Hydroxyphenylacetate

To allow for growth on Tyramine, we already introduced the metabolite 4-Hydroxyphenylacetate (C00642). Hence, if we assume that 4-Hydroxyphenylacetate can be transported into the cell, growth on this compound will also be possible.

In [30]:
# Add metabolite 4HPA and downstream pathway metabolites to the model
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00642[e]", compartment="e", name="4-Hydroxyphenylacetate")])
conditions.loc[conditions['Medium'] == 'p_Hydroxy_Phenylacetic_Acid', 'KEGG'] = 'C00642[e]' # also update in conditions DF

# Add required reactions
# r1
reaction = cobra.Reaction('Gap_4HPA_r1')
reaction.name = '4HPA uptake'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00642[e]': -1,
    'C00642[c]': 1
})

### Growth on 3-Hydroxyphenylacetate

From the niger GEM: NADPH[c] + Oxygen[c] + 3-hydroxyphenylacetic acid[c] => Water[c] + Homogentisate[c] + NADP+[c]

-> add this reaction

In [31]:
# Add metabolite 3HPA and downstream pathway metabolites to the model
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C05593[e]", compartment="e", name="3-Hydroxyphenylacetate")])
conditions.loc[conditions['Medium'] == 'm_Hydroxy_Phenylacetic_Acid', 'KEGG'] = 'C05593[e]' # also update in conditions DF
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C05593[c]", compartment="c", name="3-Hydroxyphenylacetate")])

# Add required reactions
# r1
reaction = cobra.Reaction('Gap_3HPA_r1')
reaction.name = '3HPA uptake'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C05593[e]': -1,
    'C05593[c]': 1
})

# r2
reaction = cobra.Reaction('Gap_3HPA_r2')
reaction.name = '3HPA r2'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C05593[c]': -1,
    'C00005[c]': -1,
    'C00007[c]': -1,
    'C00001[c]': 1,
    'C00006[c]': 1,
    'C00544[c]': 1,
})

### Growth on L-Norvaline

Not sure what happens, but first step could be L-Norvaline => 2-oxopentanoate + NH3 (C01826 => C06255 + C00014)
When 2-oxopentanoate is allowed to escape, we simulate metabolic activity and growth. Now, to metabolize 2-oxopentanoate, one avenue is 
2-oxopentanoate (α-ketoisovalerate) + CoA → isobutyryl-CoA + CO₂.

In KEGG, the latter would be: C06255 + C00010 =>  C00630 + C00011.
Lets see if that works.


In [32]:
# Add metabolite L-Norvaline and downstream pathway metabolites to the model
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C01826[e]", compartment="e", name="L-Norvaline")])
conditions.loc[conditions['Medium'] == 'L_Norvaline', 'KEGG'] = 'C01826[e]' # also update in conditions DF
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C01826[c]", compartment="c", name="L-Norvaline")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C01826[m]", compartment="m", name="L-Norvaline")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C06255[m]", compartment="m", name="2-oxopentanoate")])

# Add required reactions
# r1
reaction = cobra.Reaction('Gap_L_Norvaline_r1')
reaction.name = 'Norvaline uptake'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C01826[e]': -1,
    'C01826[c]': 1
})

# r2
reaction = cobra.Reaction('Gap_L_Norvaline_r2')
reaction.name = 'Norvaline gap2'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C01826[c]': -1,
    'C01826[m]': 1
})

# r3
reaction = cobra.Reaction('Gap_L_Norvaline_r3')
reaction.name = 'Norvaline gap3'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C01826[m]': -1,
    'C00014[m]': 1,
    'C06255[m]': 1
})

# r4
reaction = cobra.Reaction('Gap_L_Norvaline_r4')
reaction.name = 'Norvaline gap4'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C06255[m]': -1,
    'C00010[m]': -1,
    'C00630[m]': 1,
    'C00011[m]': 1
})

There are still seem to be many reactions to add unfortunately:

- First reaction downstream of isobutyryl is one we have: r811. However, there is a mistake in this reaction I think. Currently, it goes from isobutyryl-CoA (C00630) to 2-methylbut-2-enoyl-CoA (C03345). However, the former has 25C, while the latter has 26C. In KEGG, the compound that is formed in stead is 2-Methylprop-2-enoyl-CoA (C03460), which has 25C. Hence, we should update the reaction accordingly. Note that C03460 currently does not exist in the mitochondria, only in the peroxisome.
- Next step is C03460 to C06000 (2-Methylprop-2-enoyl-CoA + H2O <=> (S)-3-Hydroxyisobutyryl-CoA), then C06000 to C06001 (OtherAsp_R05066 but in mitochondria), then C02170 (OtherAsp_R03871 but in mitochondria and reverse), then C01213 (OtherAsp_R03383?), and then C00091. While I have many of these compounds, I dont seem to have them in the mitochondria.


In [33]:
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C03460[m]", compartment="m", name="2-Methylprop-2-enoyl-CoA")])
ensemble.base_model.reactions.get_by_id("r811").reaction = "C00016[m] + C00630[m] --> C01352[m] + C03460[m]"

# r5
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C06000[m]", compartment="m", name="(S)-3-Hydroxyisobutyryl-CoA")])
reaction = cobra.Reaction('Gap_L_Norvaline_r5')
reaction.name = 'Norvaline gap5'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C03460[m]': -1,
    'C00001[m]': -1,
    'C06000[m]': 1
})

# r6
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C06001[m]", compartment="m", name="(S)-3-Hydroxyisobutyrate")])
reaction = cobra.Reaction('Gap_L_Norvaline_r6')
reaction.name = 'Norvaline gap6'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C06000[m]': -1,
    'C00001[m]': -1,
    'C00010[m]': 1,
    'C06001[m]': 1
})

# r7
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C06002[m]", compartment="m", name="(S)-Methylmalonate semialdehyde")])
ensemble.base_model.reactions.get_by_id("OtherAsp_R05066").reaction = "C06001[m] + C00003[m] --> C06002[m] + C00004[m]"
# TODO Originally, the gene catalyzing OtherAsp_R05066 was only present in isolates Aspergillus_oryzae_NRRL_5589, Aspergillus_oryzae_NRRL_35890, and Pan_oryzae
# I will now give all members access to the gene, but this could be used to differentiate between members.
for model, bound in ensemble.features.OtherAsp_R05066_upper_bound.states.items():
    if(model in submembers):
        ensemble.features.OtherAsp_R05066_upper_bound.states.update({model: 1000})

# r8
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C02170[m]", compartment="m", name="Methylmalonate")])
ensemble.base_model.reactions.get_by_id("OtherAsp_R03871").reaction = "C00027[m] + C02170[m] <-- C00001[m] + C00007[m] + C06002[m]"
# TODO Originally, the gene catalyzing OtherAsp_R05066 was only present in Pan_oryzae
# I will now give all members access to the gene, but this could be used to differentiate between members.
for model, bound in ensemble.features.OtherAsp_R03871_lower_bound.states.items():
    if(model in submembers):
        ensemble.features.OtherAsp_R03871_lower_bound.states.update({model: -1000})

# r9
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C01213[m]", compartment="m", name="(R)-Methylmalonyl-CoA")])
# Note that we already have a reaction involving this metabolite in the cytoplasm:
# C00002[c] + C00010[c] + C02170[c] <=> C00013[c] + C00020[c] + C01213[c]
# ATP + coenzyme A + Methylmalonate <=> diphosphate + AMP + (R)-Methylmalonyl-CoA
# This reaction, however, requires ATP. I will therefore use the reaction from KEGG:
# C06002 + C00010 + C00003 <=> C01213 + C00004
# (S)-Methylmalonate semialdehyde + CoA + NAD+ <=> (R)-Methylmalonyl-CoA + NADH
reaction = cobra.Reaction('Gap_L_Norvaline_r9')
reaction.name = 'Norvaline gap9'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C06002[m]': -1,
    'C00010[m]': -1,
    'C00003[m]': -1,
    'C01213[m]': 1,
    'C00004[m]': 1
})

# r10
reaction = cobra.Reaction('Gap_L_Norvaline_r10')
reaction.name = 'Norvaline gap10'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C01213[m]': -1,
    'C00091[m]': 1
})

### Growth on gentiobiose

It is very likely that the disaccharide gentiobiose is broken down to two units of glucose by cleavage of the beta-1,6-glycoside bond. One canditdate gene in Aspergillus oryzae is AO090038000425 (cluster_2533), according to https://www.researchgate.net/publication/281351820_Purification_and_enzymatic_characterization_of_a_novel_b-16-glucosidase_from_Aspergillus_oryzae. This gene we currently assigned to the function cellobiose[c] + H2O[c] => 2 beta-D-glucose[c], reaction OtherAsp_R00026.Note, cellobiose is a disaccharide composed of two glucose molecules linked by a β(1→4) bond, and hence this is a deifferent type of cleavage. We can safely add this gene to all isolates, it is a present in all of them according to our BPGA analysis.

In [34]:
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C08240[e]", compartment="e", name="Gentiobiose")])
conditions.loc[conditions['Medium'] == 'b_Gentiobiose', 'KEGG'] = 'C08240[e]' # also update in conditions DF

# r1
reaction = cobra.Reaction('Gap_b_Gentiobiose_r1')
reaction.name = 'Gentiobiose gap1'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C08240[e]': -1,
    'C00221[e]': 2
})
reaction.gene_reaction_rule = 'cluster_2533'

### Growth on Turanose

Turanose is a disaccharide of glucose (alpha-D-glucose) and fructose (beta-D-fructose). These monosaccharides could be eleased in the presence of an enzyme that can cleave a  α(1,3) glycosidic bond. However, I do not immediately find a gene that would catalyze this reaction in Aspergillus oryzae.

In [35]:
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C19636[e]", compartment="e", name="Turanose")])
conditions.loc[conditions['Medium'] == 'Turanose', 'KEGG'] = 'C19636[e]' # also update in conditions DF

# r1
reaction = cobra.Reaction('Gap_Turanose_r1')
reaction.name = 'Turanose gap1'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C19636[e]': -1,
    'C00267[e]': 1,
    'C00095[e]': 1
})

### Growth on Palatinose

In this disaccharide, we have a α(1,6) glycosidic bond connecting glucose and fructose. We thus need a gene that is able to cleave such bonds. Another disaccharide that has this type of bond is isomaltose (cleaved by cluster_6858 and cluster_1800). We may thus assign these genes to this reaction as well (even though this is speculative).


In [36]:
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C01742[e]", compartment="e", name="Palatinose")])
conditions.loc[conditions['Medium'] == 'Palatinose', 'KEGG'] = 'C01742[e]' # also update in conditions DF

# r1
reaction = cobra.Reaction('Gap_Palatinose_r1')
reaction.name = 'Palatinose gap1'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C01742[e]': -1,
    'C00267[e]': 1,
    'C00095[e]': 1
})
# reaction.gene_reaction_rule = 'cluster_6858 or cluster_1800'
# Not all isolates have these genes, so Ill be a bit more careful to where I add it.

### Growth on beta-Cyclodextrin

Growth on beta-Cyclodextrin (C13183) probably results from cleaving the α(1,4) glycosidic bonds between the glucose units. It is no quite clear which enzyme would be responsible for that, it could be an alpha-amylase or a glucoamylase, however, they may not act on cyclic sugars. Alternatively, there is a class of cyclodextrinases, but on uniprot these are only described for 1 arachae and 16 bacteria. Hence, I will not use a gene for it for now. The reaction specification will be similar to starch[e] => alpha-D-glucose[e], as it is not quite clear which other oligosaccharides would be released in the process of cleaving of D-Glucose.

In [37]:
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C13183[e]", compartment="e", name="b_Cyclodextrin")])
conditions.loc[conditions['Medium'] == 'b_Cyclodextrin', 'KEGG'] = 'C13183[e]' # also update in conditions DF

# r1
reaction = cobra.Reaction('Gap_b_Cyclodextrin_r1')
reaction.name = 'b_Cyclodextrin gap1'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C13183[e]': -1,
    'C00267[e]': 1,
})

### Growth on myo-inositol

Perhaps the most viable route to metabolize myo-inositol is through glucuronate. From there, there are several pathways that are quite plausible.
These are great references: https://febs.onlinelibrary.wiley.com/doi/full/10.1002/1873-3468.12946 & https://www.nature.com/articles/srep26329

1. Through xylulose -> only described in animals
2. Through glycerol -> requires D-galacturonate, which cannot be reached
3. Through gluconate-6-P into PPP. This is what is more or less established in A. niger.

So, if we take the Aspergillus niger route, we get:

1. Myo-inositol => D-glucuronate (C00191[c]): r198
2. D-glucuronate => L-gulonate (C00800[c]): OtherAsp_R01481
3. L-gulonate => 2-keto-L-gulonate (C15673[c])
4. 2-keto-L-gulonate => L-idonate (C00770[c])
5. L-idonate => 5-keto-D-gluconate (D-Tagaturonate; C00558[c] !! not sure)
6. 5-keto-D-gluconate => D-gluconate (C00257[c])
7. D-gluconate => D-gluconate-6-phosphate (6-Phospho-D-gluconate, C00345[c]): r68

In [38]:
conditions.loc[conditions['Medium'] == 'myo_Inositol', 'KEGG'] = 'C00137[e]' # also update in conditions DF
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C15673[c]", compartment="c", name="2-keto-L-gulonate")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00770[c]", compartment="c", name="L-idonate")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00558[c]", compartment="c", name="5-keto-D-gluconate")])

In [39]:
# r1: L-gulonate + NAD+ => 2-keto-L-gulonate + NADH
reaction = cobra.Reaction('Gap_myo_Inositol_r1')
reaction.name = 'myo_Inositol gap1'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00800[c]': -1,
    'C00003[c]': -1,
    'C15673[c]': 1,
    'C00004[c]': 1
})

# r2: 2-keto-L-gulonate + NADH => L-idonate + NAD+
reaction = cobra.Reaction('Gap_myo_Inositol_r2')
reaction.name = 'myo_Inositol gap2'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C15673[c]': -1,
    'C00004[c]': -1,
    'C00770[c]': 1,
    'C00003[c]': 1
})

# r3: 2-keto-L-gulonate + NADPH => L-idonate + NADP+
reaction = cobra.Reaction('Gap_myo_Inositol_r3')
reaction.name = 'myo_Inositol gap3'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C15673[c]': -1,
    'C00005[c]': -1,
    'C00770[c]': 1,
    'C00006[c]': 1
})

# r4: L-idonate + NAD+ => 5-keto-D-gluconate + NADH
reaction = cobra.Reaction('Gap_myo_Inositol_r4')
reaction.name = 'myo_Inositol gap4'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00770[c]': -1,
    'C00003[c]': -1,
    'C00558[c]': 1,
    'C00004[c]': 1
})

# r5: 5-keto-D-gluconate + NADH => D-gluconate + NAD+ (not 100% sure of cofactors)
reaction = cobra.Reaction('Gap_myo_Inositol_r5')
reaction.name = 'myo_Inositol gap5'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00558[c]': -1,
    'C00004[c]': -1,
    'C00257[c]': 1,
    'C00003[c]': 1
})

### Growth on Adonitol

In the A. niger GEM, we have reactions
- NADPH[c] + D-Ribose[c] <=> NADP+[c] + Ribitol[c] (cluster_6864)
- Ribitol[e] <=> Ribitol[c]

With Ribitol a synonym of Adonitol.

We will make the assumption that this is also what is happening in A. oryzae.

In [40]:
# target = "NADP(+)"
# for r in ensemble.base_model.metabolites:
#     if r.name == target:
#         r

In [41]:
conditions.loc[conditions['Medium'] == 'Adonitol', 'KEGG'] = 'C00474[e]' # also update in conditions DF
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00474[e]", compartment="e", name="Adonitol")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00474[c]", compartment="c", name="Adonitol")])

# r1: 
reaction = cobra.Reaction('Gap_Adonitol_r1')
reaction.name = 'Adonitol uptake'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00474[e]': -1,
    'C00474[c]': 1
})

# r2: 
reaction = cobra.Reaction('Gap_Adonitol_r2')
reaction.name = 'Adonitol gap2'
reaction.gene_reaction_rule = 'cluster_6864'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00474[c]': -1,
    'C00006[c]': -1,
    'C00005[c]': 1,
    'C00121[c]': 1
})


### Growth on g-Cyclodextrin

Growth on gamma-Cyclodextrin is probably very similar to growth on b-cyclodextrin, i.e., cleaving the α(1,4) glycosidic bonds between the glucose units. The main difference is that gamma-Cyclodextrin has 8 glucoses while beta-Cyclodextrin has 7, but I am currently ignoring that anyway. Hence, I will just pro forma add this comment in the exact same way.

In [42]:
ensemble.base_model.add_metabolites([cobra.Metabolite(id="gCycDex[e]", compartment="e", name="g_Cyclodextrin")]) # Doesnt have a KEGG ID
conditions.loc[conditions['Medium'] == 'g_Cyclodextrin', 'KEGG'] = 'gCycDex[e]' # also update in conditions DF

# r1
reaction = cobra.Reaction('Gap_g_Cyclodextrin_r1')
reaction.name = 'g_Cyclodextrin gap1'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'gCycDex[e]': -1,
    'C00267[e]': 1,
})

### Growth on N-acetyl-D-glucosamine

In [43]:
conditions.loc[conditions['Medium'] == 'N_Acetyl_D_Glucosamine', 'KEGG'] = 'C00140[e]' # also update in conditions DF

### Growth on L-Ornithine

Some isolates can already grow, but NRRL_471 and RIB40 should grow as well -> gapfil!
Also, I still need an ornithine transporter. According to literature, it uses the same system as for arginine and lysine, at least in A. nidulans (https://www.microbiologyresearch.org/content/journal/micro/10.1099/00221287-92-1-89). So for our GEM, this would be cluster_2563 or cluster_6157 or cluster_6939.

In [44]:
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00077[e]", compartment="e", name="L-ornithine")])

# r1
reaction = cobra.Reaction('Gap_L_Ornithine_r1')
reaction.name = 'L_Ornithine uptake'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00077[e]': -1,
    'C00077[c]': 1,
})

# Gapfill NRRL_471: several options (OtherAsp_R00708, OtherAsp_R00245, OtherAsp_R00707, OtherAsp_R01151),
# but OtherAsp_R01151 looks most specific
ensemble.features.OtherAsp_R01151_upper_bound.states.update({'Aspergillus_oryzae_NRRL_471': 1000})

# Gapfill RIB40: same as for NRRL_471
ensemble.features.OtherAsp_R01151_upper_bound.states.update({'Aspergillus_oryzae_RIB40_CAoGD': 1000})

### Growth on putrescine

To grow on extracellular putrescine, a transporter is needed. While this could be the same transporter as for arginine and ornithine, I dont have any direct evidence for that, so I will it unspecified for now.

In [45]:
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00134[e]", compartment="e", name="Putrescine")])
# r1
reaction = cobra.Reaction('Gap_Putrescine_r1')
reaction.name = 'Putrescine uptake'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00134[e]': -1,
    'C00134[c]': 1,
})

### b_Phenylethylamine



In [46]:
# b_Phenylethylamine = C05332
conditions.loc[conditions['Medium'] == 'b_Phenylethylamine', 'KEGG'] = 'C05332[e]' # also update in conditions DF
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C05332[e]", compartment="e", name="b_Phenylethylamine")]) # Doesnt have a KEGG ID

# r1
reaction = cobra.Reaction('Gap_b_Phenylethylamine_r1')
reaction.name = 'b_Phenylethylamine uptake'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C05332[e]': -1,
    'C05332[c]': 1,
})

# - OtherAsp_R02613: H2O[c] + oxygen[c] + Phenethylamine[c] => hydrogen peroxide[c] + NH3[c] + phenylacetaldehyde[c]
# - r1799: H2O[c] + NAD(+)[c] + phenylacetaldehyde[c] <=> NADH[c] + phenylacetate[c]
# - Unclear KEGG reaction: Phenylacetic acid <=> 4-Hydroxyphenylacetate ----> added below as reaction r2
# - 4-Hydroxyphenylacetate allows for growth, we already fixed that above

# r2
reaction = cobra.Reaction('Gap_b_Phenylethylamine_r2')
reaction.name = 'b_Phenylethylamine gap2'
reaction.lower_bound = -1000
reaction.upper_bound = 1000 
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C07086[c]': -1,
    'C00642[c]': 1,
})

# With these changes, I get a good overlap between predictions and plate data (some grow, some dont).
# This could be improved further by granting access to growth NRRL_3488. Let's see if I can identify what is missing.
# Can I do it using automatic gapfilling from e.g. NRRL_2217?

# from cobra.flux_analysis import gapfill
# NRRL_2217 = ensemble.extract_member("Aspergillus_oryzae_NRRL_2217")
# NRRL_3488 = ensemble.extract_member("Aspergillus_oryzae_NRRL_3488")
# NRRL_3488.medium = medium_Climit
# NRRL_3488.reactions.get_by_id('r1901').bounds = (0, 1000.0)
# NRRL_3488.objective = {NRRL_3488.reactions.get_by_id('r2359'): 1} # optimize growth
# new_sink = NRRL_3488.add_boundary(NRRL_3488.metabolites.get_by_id("C05332[e]"), type="sink")
# NRRL_2217.medium = medium_Climit
# NRRL_2217.reactions.get_by_id('r1901').bounds = (0, 1000.0)
# NRRL_2217.objective = {NRRL_2217.reactions.get_by_id('r2359'): 1} # optimize growth
# new_sink = NRRL_2217.add_boundary(NRRL_2217.metabolites.get_by_id("C05332[e]"), type="sink")
# result = gapfill(NRRL_3488, NRRL_2217, demand_reactions=False, iterations=4)
# result

# According to gafilling, reaction OtherAsp_R02613 would fix growth for NRRL_3488.
ensemble.features.OtherAsp_R02613_upper_bound.states.update({'Aspergillus_oryzae_NRRL_3488': 1000})

### Growth on Dihydroxyacetone

Allow passive diffusion into the cell.

In [47]:
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00184[e]", compartment="e", name="glycerone")])

# r1
reaction = cobra.Reaction('Gap_Dihydroxy_Acetone_r1')
reaction.name = 'Dihydroxy_Acetone uptake'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00184[e]': -1,
    'C00184[c]': 1,
})

### Growth on Agmatine

It can already grow, but only when provided intracellularly. Currently, there is 1 reaction for extracellular agmatine, H2O[e] + agmatine[e] <=> urea[e] + putrescine[e] (cluster_9575)
There are two possible solutions to this.
- Allow agmatine transport, which currently does not happen.
- Leave agmatine[e] extracellular, and allow urea[e] transport. 

While urea transport is quite likely, I actually dont really like the above reaction being extracellular. Not that we also have this reaction intracellular from the original GEM, and the we got the extracellular one from KEGG, so its localization was predicted using deeploc.

To me, it seems most plausible that agmatine gets transported in, and the reaction only exist intracellularly. So, I will add gene cluster_9575 to the list of genes catalyzing the intracellular reaction, remove the extracellular reaction, and allow for agmatine transport (gene unspecified, although it could be similar to e.g. arginine).

In [48]:
ensemble.base_model.reactions.get_by_id("r612").gene_reaction_rule = 'cluster_9575 or (cluster_9560 and cluster_3072 and cluster_6502)'
ensemble.base_model.remove_reactions(reactions = ['OtherAsp_R01157_Extracellular'],  remove_orphans = True)
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00179[e]", compartment="e", name="agmatine")])
# r1
reaction = cobra.Reaction('Gap_Agmatine_r1')
reaction.name = 'Agmatine uptake'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00179[e]': -1,
    'C00179[c]': 1,
})

### Growth on L-Tryptophan

While I have ATP production, this is without oxygen, and on an N plate we always get ATP production.
While metabolic activity doesnt mean growth, I could instead aim to curate growth. Let's try.

In Aspergillus niger, we have a pathway for the degradation of Tryptophan through 2,3-Dihydroxybenzoate.
However, I think we have several gaps in that pathway downstream of that, which would also explain the absence of growth on 4-HBA/catechol.

It seems that the Aspergillus niger GEM can metabolize catachol, which involves many reactions, through succinylCoA. Unfortunately, I cant readily gapfil this, since I dont have a KEGG for all of these metabolites. Either I do this manually, or I do get a keggified niger GEM. Probably manually is still faster. The reactions are:

- Oxygen[c] + L-Tryptophan[c] => L-Formylkynurenine[c] (C02700) # already have it
- L-Formylkynurenine[c] + Water[c] => Formate[c] + L-Kynurenine[c] (C00328) # already have it
- Water[c] + L-Kynurenine[c] => L-Alanine[c] + Anthranilate[c] (C00108) # already have it
- Anthranilate[c] + NADPH[c] + Oxygen[c] => 2,3-Dihydroxybenzoate[c] + NADP+[c] + Ammonia[c] (C00196)
- 2,3-Dihydroxybenzoate[c] => Catechol[c] + Carbon dioxide[c] (C00090)
- Catechol[c] + Oxygen[c] => cis,cis-Muconate[c] (C02480)
- cis,cis-Muconate[c] <=> Muconolactone[c] (C14610)
- Muconolactone[c] <=> 3-Oxoadipate enol-lactone[c] (C03586)
- Water[c] + 3-Oxoadipate enol-lactone[c] <=> 3-Oxoadipate[c] (C00846)
- 3-Oxoadipate[c] <=> 3-Oxoadipate (mitochondrial)[m]
- 3-Oxoadipate (mitochondrial)[m] + Succinyl coenzyme A[m] <=> 3-Oxoadipyl-CoA[m] + Succinate (mitochondrial)[m] (C02232)
- 3-Oxoadipyl-CoA[m] + Coenzyme A[m] <=> Acetyl coenzyme A[m] + Succinyl coenzyme A[m]

Alternatively, there could be another pathway. I already have in the GEM:

- L-tryptophan[c] => CO2[c] + tryptamine[c]
- H2O[c] + oxygen[c] + tryptamine[c] => hydrogen peroxide[c] + NH3[c] + Indole-3-acetaldehyde[c]

But then it gets stuck in Indole-3-acetaldehyde[c], I dont really see a way how this component can be metabolized further.
Also, trying to complete this pathway wouldnt help in explaining growth on 4HBA. So I will take the set of reactions above. Also, I will take the genes from A. niger if possible through the link established with the BPGA table.


In [49]:
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00196[c]", compartment="c", name="2,3-Dihydroxybenzoate")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00090[c]", compartment="c", name="Catechol")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C02480[c]", compartment="c", name="cis,cis-Muconate")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C14610[c]", compartment="c", name="Muconolactone")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C03586[c]", compartment="c", name="3-Oxoadipate enol-lactone")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00846[c]", compartment="c", name="3-Oxoadipate")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00846[m]", compartment="m", name="3-Oxoadipate")])
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C02232[m]", compartment="m", name="3-Oxoadipyl-CoA")])

# r1: Anthranilate[c] + NADPH[c] + Oxygen[c] => 2,3-Dihydroxybenzoate[c] + NADP+[c] + Ammonia[c]
reaction = cobra.Reaction('Gap_L_Tryptophan_r1')
reaction.name = 'L_Tryptophan gap1'
reaction.gene_reaction_rule = 'cluster_7880'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00108[c]': -1,
    'C00005[c]': -1,
    'C00007[c]': -1,
    'C00196[c]': 1,
    'C00014[c]': 1, 
    'C00006[c]': 1
})

# r2: 2,3-Dihydroxybenzoate[c] => Catechol[c] + Carbon dioxide[c]
reaction = cobra.Reaction('Gap_L_Tryptophan_r2')
reaction.name = 'L_Tryptophan gap2'
reaction.gene_reaction_rule = 'cluster_12209'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00196[c]': -1,
    'C00090[c]': 1,
    'C00011[c]': 1
})

# r3: Catechol[c] + Oxygen[c] => cis,cis-Muconate[c]
reaction = cobra.Reaction('Gap_L_Tryptophan_r3')
reaction.name = 'L_Tryptophan gap3'
reaction.gene_reaction_rule = 'cluster_13016'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00090[c]': -1,
    'C00007[c]': 1,
    'C02480[c]': 1
})

# r4: cis,cis-Muconate[c] <=> Muconolactone[c] (C14610), gene cluster_11391
reaction = cobra.Reaction('Gap_L_Tryptophan_r4')
reaction.name = 'L_Tryptophan gap4'
reaction.gene_reaction_rule = 'cluster_11391'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C02480[c]': -1,
    'C14610[c]': 1
})

# r5: Muconolactone[c] <=> 3-Oxoadipate enol-lactone[c] (C03586)
reaction = cobra.Reaction('Gap_L_Tryptophan_r5')
reaction.name = 'L_Tryptophan gap5'
reaction.gene_reaction_rule = 'cluster_17315'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C14610[c]': -1,
    'C03586[c]': 1
})

# r6: Water[c] + 3-Oxoadipate enol-lactone[c] <=> 3-Oxoadipate[c] (C00846)
reaction = cobra.Reaction('Gap_L_Tryptophan_r6')
reaction.name = 'L_Tryptophan gap6'
reaction.gene_reaction_rule = 'cluster_7045'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00001[c]': -1,
    'C03586[c]': -1,
    'C00846[c]': 1
})

# r7: 3-Oxoadipate[c] <=> 3-Oxoadipate[m]
reaction = cobra.Reaction('Gap_L_Tryptophan_r7')
reaction.name = 'L_Tryptophan gap7'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00846[c]': -1,
    'C00846[m]': 1
})

# r8: 3-Oxoadipate (mitochondrial)[m] + Succinyl coenzyme A[m] <=> 3-Oxoadipyl-CoA[m] + Succinate (mitochondrial)[m] (C02232)
reaction = cobra.Reaction('Gap_L_Tryptophan_r8')
reaction.name = 'L_Tryptophan gap8'
reaction.gene_reaction_rule = 'cluster_1445 or cluster_356'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00846[m]': -1,
    'C00091[m]': -1,
    'C02232[m]': 1,
    'C00042[m]': 1
})

# r9: 3-Oxoadipyl-CoA[m] + Coenzyme A[m] <=> Acetyl coenzyme A[m] + Succinyl coenzyme A[m] (1184968 or 1146917 or 1098182 or 1181771)
reaction = cobra.Reaction('Gap_L_Tryptophan_r9')
reaction.name = 'L_Tryptophan gap9'
reaction.gene_reaction_rule = 'cluster_1212 or cluster_5014 or cluster_9581'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C02232[m]': -1,
    'C00010[m]': -1,
    'C00024[m]': 1,
    'C00091[m]': 1
})

In [50]:
# Import BPGA table 
import pandas as pd
BPGA = pd.read_csv('BPGA2ortho_GEM_custom.csv', delimiter=";", dtype=str)

In [51]:
BPGA[BPGA['niger'].str.contains('1183763', regex=True, na=False)].iloc[:, :2]

,cluster,present_in_n_genomes
6092,cluster_7127,159


### Growth on 4_Hydroxy_Benzoic_Acid

From the literature and the niger GEM, we will assume that 4-hydroxybenzoate[c] can diffuse passively through the cell membrane.

In [52]:
ensemble.base_model.add_metabolites([cobra.Metabolite(id="C00156[e]", compartment="e", name="4_Hydroxy_Benzoic_Acid")]) # Doesnt have a KEGG ID
conditions.loc[conditions['Medium'] == '4_Hydroxy_Benzoic_Acid', 'KEGG'] = 'C00156[e]' # also update in conditions DF

# r1
reaction = cobra.Reaction('Gap_4HBA_r1')
reaction.name = '4HBA uptake'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00156[e]': -1,
    'C00156[c]': 1
})

# r2: 3,4-dihydroxybenzoate[c] to catechol[c] + CO2[c]
reaction = cobra.Reaction('Gap_4HBA_r2')
reaction.name = '4HBA gap2'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00156[c]': -1,
    'C00090[c]': 1,
    'C00011[c]': 1
})

### Growth on Xanthine

The most sensible pathway seems to move to urea and glyoxylate. I think all (or most) of the reactions are there, but still there doesnt seem to be growth.
So lets see whats wrong.

- OtherAsp_R02107 ; H2O[c] + oxygen[c] + xanthine[c] => hydrogen peroxide[c] + urate[c] ; cluster_1078

Gene AO090011000588 catalyzes the next step from urate to 5-hydroxyisourate.

Urate + Oxygen + H2O <=> 5-Hydroxyisourate + Hydrogen peroxide

In [53]:
# target = "oxygen"
# for r in ensemble.base_model.metabolites:
#     if r.name == target:
#         r

In [54]:
# r1: Urate + Oxygen + H2O <=> 5-Hydroxyisourate + Hydrogen peroxide
reaction = cobra.Reaction('Gap_Xanthine_r1')
reaction.name = 'Xanthine gap1'
reaction.gene_reaction_rule = 'cluster_13282'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00366[c]': -1,
    'C00007[c]': -1,
    'C00001[c]': -1,
    'C11821[c]': 1,
    'C00027[c]': 1
})

# I do think this should work, but the fact it doesnt probably means we have a dead end metabolite.
# From what I can see, the main candidate would be glyoxylate[p].
# Perhaps adding a reaction from niger could help
# r2832: Water[p] + Acetyl coenzyme A[p] + Glyoxylate[p] => Coenzyme A[p] + (S)-Malate[p] (gene 1183763)
# Test by letting Glyoxylate[p] out first

reaction = cobra.Reaction('Gap_Xanthine_r2')
reaction.name = 'Xanthine gap2'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00048[p]': -1,
    'C00048[c]': 1
})

reaction = cobra.Reaction('Gap_Xanthine_r3')
reaction.name = 'Xanthine gap3'
reaction.gene_reaction_rule = 'cluster_7127'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C00001[c]': -1,
    'C00024[c]': -1,
    'C00048[c]': -1,
    'C00010[c]': 1,
    'C00149[c]': 1
})


In [57]:
ensemble.base_model.reactions.get_by_id('r1901').bounds = (0, 1000.0)
ensemble.base_model.objective = {ensemble.base_model.reactions.get_by_id('r2359'): 1} # optimize growth
objective_id = 'r2359'

# Select the desired minimal medium
ensemble.base_model.medium = medium_Nlimit.copy()

new_sink = ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id("C00385[c]"), type="sink")
# new_demand = ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id("C00010[p]"), type="demand")
fluxes = flux_balance.optimize_ensemble(ensemble,
                                        specific_models = submembers,
                                        return_flux = ['r2359', 'r2199', 'r2200', 'r2202', 'r2203',
                                                       'r2204', 'r2205', 'r2298', 'r2317', 'r1901',
                                                       'r2333', 'r2339', 'r2357', 'r2360'])
fluxes
ensemble.base_model.remove_reactions([new_sink.id])

,r2359,r2199,r2200,r2202,r2203,r2204,r2205,r2298,r2317,r1901,r2333,r2339,r2357,r2360
Pan_oryzae,47.098016,0.0,0.0,277.301919,28.957336,4.121960,1000.000000,547.808977,0.0,0.0,0.0,0.0,1000.000000,0.000000
template,32.999828,0.0,0.0,560.341783,24.279777,3.686194,594.988242,0.000000,0.0,0.0,0.0,0.0,1000.000000,60.395409
Aspergillus_oryzae_RIB40_CAoGD,33.334179,0.0,0.0,555.600950,19.727756,2.917366,594.287384,0.000000,0.0,0.0,0.0,0.0,1000.000000,56.286022
Aspergillus_oryzae_NRRL_2217,44.398748,0.0,0.0,479.729431,37.993253,6.024825,1000.000000,0.000000,0.0,0.0,0.0,0.0,1000.000000,76.424589
Aspergillus_oryzae_NRRL_3483,44.237300,0.0,0.0,410.494278,37.855097,6.002917,630.790896,0.000000,0.0,0.0,0.0,0.0,1000.000000,76.146685
Aspergillus_oryzae_NRRL_5589,44.267131,0.0,0.0,410.502408,37.880625,6.006965,630.879094,0.000000,0.0,0.0,0.0,0.0,1000.000000,76.198034
Aspergillus_oryzae_NRRL_3488,44.137006,0.0,0.0,410.466943,37.769273,5.989307,630.494368,0.000000,0.0,0.0,0.0,0.0,1000.000000,75.974046
Aspergillus_oryzae_NRRL_5592,44.137006,0.0,0.0,410.466943,37.769273,5.989307,630.494368,0.000000,0.0,0.0,0.0,0.0,1000.000000,75.974046
Aspergillus_oryzae_NRRL_35890,44.297721,0.0,0.0,410.510746,37.906801,6.011116,847.219433,0.000000,0.0,0.0,0.0,0.0,1000.000000,76.250689
Aspergillus_oryzae_NRRL_471,44.137006,0.0,0.0,410.466943,37.769273,5.989307,1000.000000,0.000000,0.0,0.0,0.0,0.0,973.501119,75.974046


### Growth on sorbic acid

Suggestion, via hexanoate, and then FA degradation. But, FA biosynthesis occurs in the cytoplasm, and degradation in the mitochondria, and is unclear to me if/how this is currently connected. Also, a reaction from sorbic acid (hexadienoate) to hexanoate is needed, i.e,

Sorbic acid (2,4-hexadienoic acid) + NADH (or NADPH) → Hexanoic acid (hexanoate) + NAD⁺ (or NADP⁺)

With regard to FA synthesis and degradation. We only need the degradation part here. So, we might as well assume that sorbic acid can diffuse or be transported to the mitochondria.
Note, there are in theory other mechanisms by which sorbic acid may be metabolized. I here consider conversion to hexanoate and subsequent FA degradation as the most plausible.

TODO: the pathway I ahve now does not sustain growth. AcetylCoA can (if all went well) be produced, but when entering TCA, it needs to combine with oxoglutarate to form citrate (which does support growth). But to get oxoglutarate is not directly feasible, unless AcetylCoA can be converted to pyruvate, but this is unlikely in oryzae.

In [ ]:
ensemble.base_model.add_metabolites([cobra.Metabolite(id="Sorbac[e]", compartment="e", name="Sorbic_Acid")]) # Doesnt have a KEGG ID
conditions.loc[conditions['Medium'] == 'Sorbic_Acid', 'KEGG'] = 'Sorbac[e]'
ensemble.base_model.add_metabolites([cobra.Metabolite(id="Sorbac[c]", compartment="c", name="Sorbic_Acid")]) # Doesnt have a KEGG ID

# r1 uptake, directly to mitochondria
reaction = cobra.Reaction('Gap_Sorbac_r1')
reaction.name = 'Sorbic_Acid uptake'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'Sorbac[e]': -1,
    'Sorbac[c]': 1
})

# r2: Sorbic_acid + NADH → Hexanoate + NAD⁺
reaction = cobra.Reaction('Gap_Sorbac_r2')
reaction.name = 'Sorbic_Acid gap2'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'Sorbac[c]': -1,
    'C00004[c]': -1,
    'C01585[c]': 1,
    'C00003[c]': 1
})

# r3 after hexanoate -> hexanoylCoA, hexanoylCoA must be able to go to the mitochondria, which given its short length can be passive
reaction = cobra.Reaction('Gap_Sorbac_r3')
reaction.name = 'Sorbic_Acid gap3'
ensemble.base_model.add_reactions([reaction])
reaction.add_metabolites({
    'C05270[c]': -1,
    'C05270[m]': 1
})

In [ ]:
# # Check if growth on C05270[m] is supported.
# ensemble.base_model.reactions.get_by_id('r1901').bounds = (0, 1000.0)
# ensemble.base_model.objective = {ensemble.base_model.reactions.get_by_id('r2359'): 1} # optimize growth
# objective_id = 'r2359'

# ensemble.base_model.medium = medium_Climit.copy()

# # Try if it grows with extracellular sink
# new_sink = ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id("C00158[m]"), type="sink") #  	C00024[m]

# fluxes = flux_balance.optimize_ensemble(ensemble,
#                                         specific_models = submembers,
#                                             return_flux = ['r2359', 'r2199', 'r2200', 'r2202', 'r2203',
#                                                            'r2204', 'r2205', 'r2298', 'r2317', 'r1901',
#                                                            'r2333', 'r2339', 'r2357', 'r2360'])
# fluxes
# # ensemble.base_model.remove_reactions([new_sink.id])

,r2359,r2199,r2200,r2202,r2203,r2204,r2205,r2298,r2317,r1901,r2333,r2339,r2357,r2360
Pan_oryzae,39.862332,0.0,232.453573,7.991843,18.844616,3.488702,0.0,0.0,0.0,0.0,0.000000,0.000000,659.110031,0.0
template,18.398274,0.0,113.771748,252.540103,13.536616,2.055150,0.0,0.0,0.0,0.0,33.036827,360.178310,971.731058,0.0
Aspergillus_oryzae_RIB40_CAoGD,18.703959,0.0,103.589800,250.538340,8.842155,1.636947,0.0,0.0,0.0,0.0,35.222214,368.516670,977.276120,0.0
Aspergillus_oryzae_NRRL_2217,26.048778,0.0,168.581527,391.347210,22.290670,3.534769,0.0,0.0,0.0,0.0,281.918374,429.896965,1000.000000,0.0
Aspergillus_oryzae_NRRL_3483,26.033382,0.0,168.481887,391.352064,22.277495,3.532680,0.0,0.0,0.0,0.0,282.461526,429.731272,1000.000000,0.0
Aspergillus_oryzae_NRRL_5589,25.982814,0.0,168.154620,391.755913,22.234222,3.525818,0.0,0.0,0.0,0.0,283.857598,429.381007,1000.000000,0.0
Aspergillus_oryzae_NRRL_3488,25.982814,0.0,168.154620,391.755913,22.234222,3.525818,0.0,0.0,0.0,0.0,283.857598,429.381007,1000.000000,0.0
Aspergillus_oryzae_NRRL_5592,25.991879,0.0,168.213286,390.981226,22.241979,3.527048,0.0,0.0,0.0,0.0,275.409254,425.761496,1000.000000,0.0
Aspergillus_oryzae_NRRL_35890,25.998150,0.0,168.253873,391.363171,22.247346,3.527899,0.0,0.0,0.0,0.0,283.704462,429.339324,1000.000000,0.0
Aspergillus_oryzae_NRRL_471,25.982814,0.0,168.154620,391.755913,22.234222,3.525818,0.0,0.0,0.0,0.0,283.857598,429.381007,1000.000000,0.0


In [ ]:
# ensemble.base_model.remove_reactions([new_sink.id])

# Run simulations

## Select target metabolites

In [60]:
conditions['basePlus150'] = pd.to_numeric(conditions['basePlus150'])
conditions['basePlus100'] = pd.to_numeric(conditions['basePlus100'])

conditions_target = conditions[
    (conditions['basePlus150'] >= 3) |
    ((conditions['basePlus150'] >= 2) & (conditions['basePlus100'] >= 4))
]
# conditions_target = conditions.iloc[conditions.basePlus150.values >= 3]
conditions_target

,Plate,Well,Medium,Condition,basePlus100,basePlus150,KEGG
1,PM01,A02,L_Arabinose,L_Arabinose_PM01,8,8,C00259[e]
19,PM01,B08,D_Xylose,D_Xylose_PM01,8,8,C00181[e]
27,PM01,C04,D_Ribose,D_Ribose_PM01,8,8,C00121[e]
89,PM01,H06,L_Lyxose,L_Lyxose_PM01,8,8,C01508[e]
115,PM02A,B08,Arbutin,Arbutin_PM02A,8,8,C06186[e]
276,PM03B,H01,Ala_Asp,Ala_Asp_PM03B,8,8,Ala_Asp[e]
215,PM03B,B12,L_Tryptophan,L_Tryptophan_PM03B,8,7,C00078[e]
287,PM03B,H12,Met_Ala,Met_Ala_PM03B,8,7,Met_Ala[e]
188,PM02A,H09,Dihydroxy_Acetone,Dihydroxy_Acetone_PM02A,8,6,C00184[c]
29,PM01,C06,L_Rhamnose,L_Rhamnose_PM01,7,6,C00507[e]


In [61]:
conditions_target = conditions_target.dropna(subset = ['KEGG'])
conditions_target = conditions_target.rename(columns = {'KEGG': 'Metabolite'})
conditions_target

,Plate,Well,Medium,Condition,basePlus100,basePlus150,Metabolite
1,PM01,A02,L_Arabinose,L_Arabinose_PM01,8,8,C00259[e]
19,PM01,B08,D_Xylose,D_Xylose_PM01,8,8,C00181[e]
27,PM01,C04,D_Ribose,D_Ribose_PM01,8,8,C00121[e]
89,PM01,H06,L_Lyxose,L_Lyxose_PM01,8,8,C01508[e]
115,PM02A,B08,Arbutin,Arbutin_PM02A,8,8,C06186[e]
276,PM03B,H01,Ala_Asp,Ala_Asp_PM03B,8,8,Ala_Asp[e]
215,PM03B,B12,L_Tryptophan,L_Tryptophan_PM03B,8,7,C00078[e]
287,PM03B,H12,Met_Ala,Met_Ala_PM03B,8,7,Met_Ala[e]
188,PM02A,H09,Dihydroxy_Acetone,Dihydroxy_Acetone_PM02A,8,6,C00184[c]
29,PM01,C06,L_Rhamnose,L_Rhamnose_PM01,7,6,C00507[e]


In [62]:
medium_Climit
medium_Nlimit

{'r2199': 0,
 'r2200': 1000.0,
 'r2202': 1000.0,
 'r2203': 1000.0,
 'r2204': 1000.0,
 'r2205': 0}

{'r2199': 0,
 'r2200': 0,
 'r2202': 1000.0,
 'r2203': 1000.0,
 'r2204': 1000.0,
 'r2205': 1000.0}

## Objective growth

In [63]:
ensemble.base_model.reactions.get_by_id('r1901').bounds = (0, 1000.0)
ensemble.base_model.objective = {ensemble.base_model.reactions.get_by_id('r2359'): 1} # optimize growth
objective_id = 'r2359'

In [64]:
import re
import numpy as np
np.random.seed(333)

met_ids = [met.id for met in ensemble.base_model.metabolites] # convenient
# Use regex to remove brackets and the letter inside
conditions_target['Metabolite_trunc'] = [re.sub(r'\[.\]', '', s) for s in conditions_target.Metabolite]

growth = []
oxygen = []
sink = []

for i in range(len(conditions_target.index)):

    # Select the desired minimal medium
    if(conditions_target['Plate'].iloc[i] != "PM03B"):
        ensemble.base_model.medium = medium_Climit.copy()
    else:
        ensemble.base_model.medium = medium_Nlimit.copy()

    # if the metabolite exists extracellularly
    if(conditions_target['Metabolite_trunc'].iloc[i] + "[e]" in met_ids):
        # Try if it grows with extracellular sink
        new_sink = ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id(conditions_target['Metabolite_trunc'].iloc[i] + "[e]"), type="sink")
        fluxes = flux_balance.optimize_ensemble(ensemble,
                                                specific_models = submembers,
                                                  return_flux = ['r2359', 'r2199', 'r2200', 'r2202', 'r2203',
                                                                 'r2204', 'r2205', 'r2298', 'r2317', 'r1901',
                                                                 'r2333', 'r2339', 'r2357', 'r2360']) 
        ensemble.base_model.remove_reactions([new_sink.id])

        if(any(fluxes[objective_id] > 0.001)): # if at least 1 isolate grew (including the pan-oryzae model), save results
            sink.append("Extracellular")
            growth.append(fluxes[objective_id].values)
            oxygen.append(fluxes['r2202'].values)
        else: # if none of the isolates grew:
            if(conditions_target['Metabolite_trunc'].iloc[i] + "[c]" in met_ids): # if the metabolite exists intracellularly as well:
                new_sink = ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id(conditions_target['Metabolite_trunc'].iloc[i] + "[c]"), type="sink")
                fluxes = flux_balance.optimize_ensemble(ensemble,
                                                specific_models = submembers,
                                                  return_flux = ['r2359', 'r2199', 'r2200', 'r2202', 'r2203',
                                                                 'r2204', 'r2205', 'r2298', 'r2317', 'r1901',
                                                                 'r2333', 'r2339', 'r2357', 'r2360']) 
                ensemble.base_model.remove_reactions([new_sink.id])
                if(any(fluxes[objective_id] > 0.001)): # if at least 1 isolate grew (including the pan-oryzae model), save results
                    sink.append("Extracellular_Intracellular")
                    growth.append(fluxes[objective_id].values)
                    oxygen.append(fluxes['r2202'].values)
                else: # else, no growth possible
                    growth.append(np.zeros(len(submembers)))
                    oxygen.append(np.zeros(len(submembers)))
                    sink.append("Extracellular_Intracellular")
            else: # if the metabolite does not exist intracellularly and it failed to grow extracellulary, no growth possible
                growth.append(np.zeros(len(submembers)))
                oxygen.append(np.zeros(len(submembers)))
                sink.append("Extracellular")
    
    # if the metabolite exists intracellularly but not extracellularly
    elif(conditions_target['Metabolite_trunc'].iloc[i] + "[c]" in met_ids):
        new_sink = ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id(conditions_target['Metabolite_trunc'].iloc[i] + "[c]"), type="sink")
        fluxes = flux_balance.optimize_ensemble(ensemble,
                                                specific_models = submembers,
                                                  return_flux = ['r2359', 'r2199', 'r2200', 'r2202', 'r2203',
                                                                 'r2204', 'r2205', 'r2298', 'r2317', 'r1901',
                                                                 'r2333', 'r2339', 'r2357', 'r2360']) 
        ensemble.base_model.remove_reactions([new_sink.id])
        sink.append("Intracellular")
        growth.append(fluxes[objective_id].values)
        oxygen.append(fluxes['r2202'].values)
    else:
        print(conditions_target['Metabolite_trunc'].iloc[i])
        growth.append(np.zeros(len(submembers)))
        oxygen.append(np.zeros(len(submembers)))
        sink.append("Metabolite_not_in_GEM")

### Wrangle results

In [65]:
# Convert the results list into a DataFrame
growth_df = pd.DataFrame(growth).T  # Transpose to make each iteration a column
growth_df.columns = conditions_target.Medium + "-" + conditions_target.Plate
growth_df['Isolate'] = submembers
# Reshape the data from wide to long format
growth_df = pd.melt(growth_df, id_vars=['Isolate'],
                    var_name='Condition', value_name='Growth')

# Convert the results list into a DataFrame
oxygen_df = pd.DataFrame(oxygen).T  # Transpose to make each iteration a column
oxygen_df.columns = conditions_target.Medium + "-" + conditions_target.Plate
oxygen_df['Isolate'] = submembers
# Reshape the data from wide to long format
oxygen_df = pd.melt(oxygen_df, id_vars=['Isolate'],
                    var_name='Condition', value_name='Oxygen_growth')

growth_df['Oxygen_growth'] = oxygen_df.Oxygen_growth
growth_df[['Medium', 'Plate']] = growth_df['Condition'].str.split('-', expand=True)

# Add medium to sink location to avoid mistakes
sink = pd.DataFrame(sink, columns=['Location_growth'])
sink['Medium'] = [med for med in conditions_target.Medium]

# Merge the dataframes on the 'Medium' column
growth_df = pd.merge(growth_df, sink[['Medium', 'Location_growth']], on='Medium', how='left')
growth_df


,Isolate,Condition,Growth,Oxygen_growth,Medium,Plate,Location_growth
0,Pan_oryzae,L_Arabinose-PM01,44.903436,405.283518,L_Arabinose,PM01,Extracellular
1,template,L_Arabinose-PM01,32.814374,284.880440,L_Arabinose,PM01,Extracellular
2,Aspergillus_oryzae_RIB40_CAoGD,L_Arabinose-PM01,33.213089,284.356501,L_Arabinose,PM01,Extracellular
3,Aspergillus_oryzae_NRRL_2217,L_Arabinose-PM01,42.851761,412.237658,L_Arabinose,PM01,Extracellular
4,Aspergillus_oryzae_NRRL_3483,L_Arabinose-PM01,42.821198,412.246896,L_Arabinose,PM01,Extracellular
...,...,...,...,...,...,...,...
445,Aspergillus_oryzae_NRRL_5589,4_Hydroxy_Benzoic_Acid-PM02A,26.285277,575.190636,4_Hydroxy_Benzoic_Acid,PM02A,Extracellular
446,Aspergillus_oryzae_NRRL_3488,4_Hydroxy_Benzoic_Acid-PM02A,26.285277,575.190636,4_Hydroxy_Benzoic_Acid,PM02A,Extracellular
447,Aspergillus_oryzae_NRRL_5592,4_Hydroxy_Benzoic_Acid-PM02A,26.801326,562.418337,4_Hydroxy_Benzoic_Acid,PM02A,Extracellular
448,Aspergillus_oryzae_NRRL_35890,4_Hydroxy_Benzoic_Acid-PM02A,26.302550,418.263342,4_Hydroxy_Benzoic_Acid,PM02A,Extracellular


## Objective ATP

In [66]:
# Objective function:
ensemble.base_model.reactions.get_by_id('r1901').bounds = (0, 1000.0)
ensemble.base_model.objective = {ensemble.base_model.reactions.get_by_id('r1901'): 1} # optimize ATP hydrolysis
objective_id = 'r1901'

In [67]:
import re
import numpy as np
np.random.seed(333)

met_ids = [met.id for met in ensemble.base_model.metabolites] # convenient
# Use regex to remove brackets and the letter inside
conditions_target['Metabolite_trunc'] = [re.sub(r'\[.\]', '', s) for s in conditions_target.Metabolite]

ATP = []
oxygen = []
sink = []

for i in range(len(conditions_target.index)):

    # Select the desired minimal medium
    if(conditions_target['Plate'].iloc[i] != "PM03B"):
        ensemble.base_model.medium = medium_Climit.copy()
    else:
        ensemble.base_model.medium = medium_Nlimit.copy()

    # if the metabolite exists extracellularly
    if(conditions_target['Metabolite_trunc'].iloc[i] + "[e]" in met_ids):
        # Try if it grows with extracellular sink
        new_sink = ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id(conditions_target['Metabolite_trunc'].iloc[i] + "[e]"), type="sink")
        fluxes = flux_balance.optimize_ensemble(ensemble,
                                                specific_models = submembers,
                                                  return_flux = ['r2359', 'r2199', 'r2200', 'r2202', 'r2203',
                                                                 'r2204', 'r2205', 'r2298', 'r2317', 'r1901',
                                                                 'r2333', 'r2339', 'r2357', 'r2360']) 
        ensemble.base_model.remove_reactions([new_sink.id])

        if(any(fluxes[objective_id] > 0.001)): # if at least 1 isolate grew (including the pan-oryzae model), save results
            sink.append("Extracellular")
            ATP.append(fluxes[objective_id].values)
            oxygen.append(fluxes['r2202'].values)
        else: # if none of the isolates grew:
            if(conditions_target['Metabolite_trunc'].iloc[i] + "[c]" in met_ids): # if the metabolite exists intracellularly as well:
                new_sink = ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id(conditions_target['Metabolite_trunc'].iloc[i] + "[c]"), type="sink")
                fluxes = flux_balance.optimize_ensemble(ensemble,
                                                specific_models = submembers,
                                                  return_flux = ['r2359', 'r2199', 'r2200', 'r2202', 'r2203',
                                                                 'r2204', 'r2205', 'r2298', 'r2317', 'r1901',
                                                                 'r2333', 'r2339', 'r2357', 'r2360']) 
                ensemble.base_model.remove_reactions([new_sink.id])
                if(any(fluxes[objective_id] > 0.001)): # if at least 1 isolate grew (including the pan-oryzae model), save results
                    sink.append("Extracellular_Intracellular")
                    ATP.append(fluxes[objective_id].values)
                    oxygen.append(fluxes['r2202'].values)
                else: # else, no ATP hydrolysis possible
                    ATP.append(np.zeros(len(submembers)))
                    oxygen.append(np.zeros(len(submembers)))
                    sink.append("Extracellular_Intracellular")
            else: # if the metabolite does not exist intracellularly and it failed to grow extracellulary, no ATP hydrolysis possible
                ATP.append(np.zeros(len(submembers)))
                oxygen.append(np.zeros(len(submembers)))
                sink.append("Extracellular")
    
    # if the metabolite exists intracellularly but not extracellularly
    elif(conditions_target['Metabolite_trunc'].iloc[i] + "[c]" in met_ids):
        new_sink = ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id(conditions_target['Metabolite_trunc'].iloc[i] + "[c]"), type="sink")
        fluxes = flux_balance.optimize_ensemble(ensemble,
                                                specific_models = submembers,
                                                  return_flux = ['r2359', 'r2199', 'r2200', 'r2202', 'r2203',
                                                                 'r2204', 'r2205', 'r2298', 'r2317', 'r1901',
                                                                 'r2333', 'r2339', 'r2357', 'r2360']) 
        ensemble.base_model.remove_reactions([new_sink.id])
        sink.append("Intracellular")
        ATP.append(fluxes[objective_id].values)
        oxygen.append(fluxes['r2202'].values)
    else:
        print(conditions_target['Metabolite_trunc'].iloc[i])
        ATP.append(np.zeros(len(submembers)))
        oxygen.append(np.zeros(len(submembers)))
        sink.append("Metabolite_not_in_GEM")

In [68]:
# Convert the results list into a DataFrame
ATP_df = pd.DataFrame(ATP).T  # Transpose to make each iteration a column
ATP_df.columns = conditions_target.Medium + "-" + conditions_target.Plate
ATP_df['Isolate'] = submembers
# Reshape the data from wide to long format
ATP_df = pd.melt(ATP_df, id_vars=['Isolate'],
                    var_name='Condition', value_name='ATP')

# Convert the results list into a DataFrame
oxygen_df = pd.DataFrame(oxygen).T  # Transpose to make each iteration a column
oxygen_df.columns = conditions_target.Medium + "-" + conditions_target.Plate
oxygen_df['Isolate'] = submembers
# Reshape the data from wide to long format
oxygen_df = pd.melt(oxygen_df, id_vars=['Isolate'],
                    var_name='Condition', value_name='Oxygen_ATP')

ATP_df['Oxygen_ATP'] = oxygen_df.Oxygen_ATP
ATP_df[['Medium', 'Plate']] = ATP_df['Condition'].str.split('-', expand=True)

# Add medium to sink location to avoid mistakes
sink = pd.DataFrame(sink, columns=['Location_ATP'])
sink['Medium'] = [med for med in conditions_target.Medium]

# Merge the dataframes on the 'Medium' column
ATP_df = pd.merge(ATP_df, sink[['Medium', 'Location_ATP']], on='Medium', how='left')
ATP_df

,Isolate,Condition,ATP,Oxygen_ATP,Medium,Plate,Location_ATP
0,Pan_oryzae,L_Arabinose-PM01,1000.0,0.000000,L_Arabinose,PM01,Extracellular
1,template,L_Arabinose-PM01,1000.0,121.928166,L_Arabinose,PM01,Extracellular
2,Aspergillus_oryzae_RIB40_CAoGD,L_Arabinose-PM01,1000.0,121.928166,L_Arabinose,PM01,Extracellular
3,Aspergillus_oryzae_NRRL_2217,L_Arabinose-PM01,1000.0,121.928166,L_Arabinose,PM01,Extracellular
4,Aspergillus_oryzae_NRRL_3483,L_Arabinose-PM01,1000.0,121.928166,L_Arabinose,PM01,Extracellular
...,...,...,...,...,...,...,...
445,Aspergillus_oryzae_NRRL_5589,4_Hydroxy_Benzoic_Acid-PM02A,1000.0,207.321429,4_Hydroxy_Benzoic_Acid,PM02A,Extracellular
446,Aspergillus_oryzae_NRRL_3488,4_Hydroxy_Benzoic_Acid-PM02A,1000.0,207.321429,4_Hydroxy_Benzoic_Acid,PM02A,Extracellular
447,Aspergillus_oryzae_NRRL_5592,4_Hydroxy_Benzoic_Acid-PM02A,1000.0,207.321429,4_Hydroxy_Benzoic_Acid,PM02A,Extracellular
448,Aspergillus_oryzae_NRRL_35890,4_Hydroxy_Benzoic_Acid-PM02A,1000.0,207.321429,4_Hydroxy_Benzoic_Acid,PM02A,Extracellular


# Export results

In [69]:
growth_df['ATP'] = ATP_df.ATP
growth_df['Oxygen_ATP'] = ATP_df.Oxygen_ATP
growth_df['Location_ATP'] = ATP_df.Location_ATP

growth_df = growth_df[['Isolate', 'Medium', 'Plate', 'Condition', 'Location_growth', 'Location_ATP', 'Growth', 'Oxygen_growth', 'ATP', 'Oxygen_ATP']]
growth_df = growth_df.rename(columns={'Plate': 'plateID'})
growth_df


,Isolate,Medium,plateID,Condition,Location_growth,Location_ATP,Growth,Oxygen_growth,ATP,Oxygen_ATP
0,Pan_oryzae,L_Arabinose,PM01,L_Arabinose-PM01,Extracellular,Extracellular,44.903436,405.283518,1000.0,0.000000
1,template,L_Arabinose,PM01,L_Arabinose-PM01,Extracellular,Extracellular,32.814374,284.880440,1000.0,121.928166
2,Aspergillus_oryzae_RIB40_CAoGD,L_Arabinose,PM01,L_Arabinose-PM01,Extracellular,Extracellular,33.213089,284.356501,1000.0,121.928166
3,Aspergillus_oryzae_NRRL_2217,L_Arabinose,PM01,L_Arabinose-PM01,Extracellular,Extracellular,42.851761,412.237658,1000.0,121.928166
4,Aspergillus_oryzae_NRRL_3483,L_Arabinose,PM01,L_Arabinose-PM01,Extracellular,Extracellular,42.821198,412.246896,1000.0,121.928166
...,...,...,...,...,...,...,...,...,...,...
445,Aspergillus_oryzae_NRRL_5589,4_Hydroxy_Benzoic_Acid,PM02A,4_Hydroxy_Benzoic_Acid-PM02A,Extracellular,Extracellular,26.285277,575.190636,1000.0,207.321429
446,Aspergillus_oryzae_NRRL_3488,4_Hydroxy_Benzoic_Acid,PM02A,4_Hydroxy_Benzoic_Acid-PM02A,Extracellular,Extracellular,26.285277,575.190636,1000.0,207.321429
447,Aspergillus_oryzae_NRRL_5592,4_Hydroxy_Benzoic_Acid,PM02A,4_Hydroxy_Benzoic_Acid-PM02A,Extracellular,Extracellular,26.801326,562.418337,1000.0,207.321429
448,Aspergillus_oryzae_NRRL_35890,4_Hydroxy_Benzoic_Acid,PM02A,4_Hydroxy_Benzoic_Acid-PM02A,Extracellular,Extracellular,26.302550,418.263342,1000.0,207.321429


In [ ]:
# For now, export to csv and inspect in R
growth_df.to_csv("simulation_results.csv")

# Downstream


## L-Proline

In [ ]:
# Gene cluster_676 is required for growth on L-Proline.
# According to our data, for the 8 isolates, only NRRL_2217 and NRRL_3483 have this gene.
# As such, predictions can be improved by turning it on in RIB40 and NRRL_35890
# But to be honest, all the other ones are NA, and on the plots they look to be
# actually growing, so might be best to just allow that reaction everywhere.

In [34]:
for model, bound in ensemble.features.OtherAsp_R03635_lower_bound.states.items():
    if(model in submembers):
        if(bound != 0):
            print(f"Model: {model}, Bound: {bound}")

Model: Aspergillus_oryzae_NRRL_35890, Bound: -1000.0
Model: Aspergillus_oryzae_NRRL_2217, Bound: -1000.0
Model: Pan_oryzae, Bound: -1000.0


In [35]:
for model, bound in ensemble.features.OtherAsp_R03635_upper_bound.states.items():
    if(model in submembers):
        if(bound != 0):
            print(f"Model: {model}, Bound: {bound}")

Model: Aspergillus_oryzae_NRRL_35890, Bound: 1000.0
Model: Aspergillus_oryzae_NRRL_2217, Bound: 1000.0
Model: Pan_oryzae, Bound: 1000.0


## Arginine, L-ornithine, and putrescine

In [ ]:
# Currently, I only get growth in a few isolates when supplementing L-ornithin intracellularly.
# It surprises me that these isolates then don't grow on Arginine or Putrescine.
# The most convincing isolate with growth on the plates is NRRL_2217
# C00062 Arginine
# C00077 Ornithine
# C00134 Putrescine
# C00086 Urea
model_NRRL_2217 = ensemble.extract_member("Aspergillus_oryzae_NRRL_2217")

model_NRRL_2217.reactions.get_by_id("r2095").bounds = (-1000,1000)

new_sink = model_NRRL_2217.add_boundary(model_NRRL_2217.metabolites.get_by_id("C00062[c]"), type="sink")
# new_sink = model_NRRL_2217.add_boundary(model_NRRL_2217.metabolites.get_by_id("C00014[e]"), type="sink") # give it extracellular NH3 => haha doesnt even use it
new_sink = model_NRRL_2217.add_boundary(model_NRRL_2217.metabolites.get_by_id("C00014[e]"), type="demand") 
# new_sink = model_NRRL_2217.add_boundary(model_NRRL_2217.metabolites.get_by_id("C00011[e]"), type="demand") 

model_NRRL_2217.reactions.get_by_id("r2205").bounds = (0,0) # do not give it glucose
solution1 = model_NRRL_2217.optimize()
solution1.objective_value
# model_NRRL_2217.remove_reactions(["SK_C00077[c]"])
# model_NRRL_2217.remove_reactions(["SK_C00014[e]"])

0.0

In [67]:
# Access fluxes
fluxes = solution1.fluxes

# Filter for active reactions (flux > 1e-6)
active_reactions = {rxn: flux for rxn, flux in fluxes.items() if flux > 0.1}
len(active_reactions)

# Display active reactions and their fluxes
for rxn, flux in active_reactions.items():
    print(f"Reaction: {rxn}, Flux: {flux}")

258

Reaction: r32, Flux: 63.78273917898282
Reaction: r33, Flux: 177.70542407352204
Reaction: r48, Flux: 28.92438970199891
Reaction: r54, Flux: 13.32335551641859
Reaction: r91, Flux: 252.5916040276234
Reaction: r96, Flux: 252.59160402762336
Reaction: r97, Flux: 253.74800275340465
Reaction: r100, Flux: 261.8949414013022
Reaction: r101, Flux: 221.13531086707434
Reaction: r102, Flux: 3.2231311203431954
Reaction: r104, Flux: 59.90196078490724
Reaction: r107, Flux: 8.208648088437485
Reaction: r110, Flux: 8.208648088434494
Reaction: r143, Flux: 10.415531997105985
Reaction: r149, Flux: 23.22499598207533
Reaction: r161, Flux: 37.53649941388467
Reaction: r192, Flux: 20.852922073937737
Reaction: r262, Flux: 35.78523229626715
Reaction: r263, Flux: 35.78523229626714
Reaction: r289, Flux: 5.03350384450115
Reaction: r333, Flux: 9.617052603854235
Reaction: r336, Flux: 35.73521107023076
Reaction: r427, Flux: 9.617052603854235
Reaction: r496, Flux: 497.9133082397871
Reaction: r513, Flux: 751.6613109931918
R

In [ ]:
model_NRRL_35890 = ensemble.extract_member("Aspergillus_oryzae_NRRL_35890")

# Add reaction of gene cluster_676, which catalyzes L-glutamate 5-semialdehyde --> L-glutamate
reaction_OtherAsp_R00245 = deepcopy(ensemble.base_model.reactions.get_by_id("OtherAsp_R00245")) # deepcopy required
model_NRRL_35890.add_reactions([reaction_OtherAsp_R00245])
model_NRRL_35890.reactions.get_by_id('OtherAsp_R00245').bounds = (0,1000)
model_NRRL_35890.reactions.get_by_id('OtherAsp_R00245')

new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id("C00148[e]"), type="sink") # give it extracellular proline
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id("C00014[e]"), type="sink") # give it extracellular NH3
model_NRRL_35890.reactions.get_by_id("r2205").bounds = (0,0) # do not give it glucose
model_NRRL_35890.reactions.get_by_id("OtherAsp_R01248").bounds = (-1000,1000) # make reversible (gibbs = 44), catalyzes L-proline <=> (S)-1-pyrroline-5-carboxylate
model_NRRL_35890.reactions.get_by_id("OtherAsp_R01251").bounds = (-1000,1000) # make reversible (gibbs = 44), catalyzes L-proline <=> (S)-1-pyrroline-5-carboxylate

solution1 = model_NRRL_35890.optimize()
solution1.objective_value
loopless1 = loopless_solution(model_NRRL_35890)
loopless1.objective_value

In [ ]:
from cobra.flux_analysis import gapfill
Pan_oryzae = ensemble.extract_member("Pan_oryzae")
model_NRRL_35890 = ensemble.extract_member("Aspergillus_oryzae_NRRL_35890")

new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id("C00148[e]"), type="sink") # give it extracellular proline
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id("C00014[e]"), type="sink") # give it extracellular NH3
model_NRRL_35890.reactions.get_by_id("r2205").bounds = (0,0) # do not give it glucose
model_NRRL_35890.objective = {model_NRRL_35890.reactions.get_by_id('r2359'): 1} # optimize growth
result = gapfill(model_NRRL_35890, Pan_oryzae, demand_reactions=True, iterations=4) # INFEASIBLE
# cant do it using gapfill???? Do it manually? Why wouldnt it work? If it works manually, what I do wrong in automatic?????


In [ ]:
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id("C00148[e]"), type="sink") # give it extracellular proline
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id("C00014[e]"), type="sink") # give it extracellular NH3
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id("C00003[c]"), type="sink") # give it extracellular NH3
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id("C00004[c]"), type="demand") # give it extracellular NH3
model_NRRL_35890.reactions.get_by_id("r2205").bounds = (0,0) # do not give it glucose
model_NRRL_35890.objective = {model_NRRL_35890.reactions.get_by_id('r2359'): 1} # optimize growth
solution1 = model_NRRL_35890.optimize()
solution1.objective_value

In [ ]:
ensemble.base_model.reactions.get_by_id("OtherAsp_R00245")

In [ ]:
model_NRRL_35890.reactions.get_by_id("OtherAsp_R00245")

In [ ]:
model_NRRL_35890.reactions.get_by_id("OtherAsp_R01248") # these reactions may very well be reversible. Gibbs is 44.
model_NRRL_35890.reactions.get_by_id("OtherAsp_R01251")

In [ ]:
# how the hell can it not grow???????
model_NRRL_35890.metabolites.get_by_id("C03912[c]")

In [ ]:
model_NRRL_35890 = ensemble.extract_member("Aspergillus_oryzae_NRRL_35890")

# Add reaction of gene cluster_676, which catalyzes L-glutamate 5-semialdehyde --> L-glutamate
reaction_OtherAsp_R00245 = deepcopy(ensemble.base_model.reactions.get_by_id("OtherAsp_R00245")) # deepcopy required
model_NRRL_35890.add_reactions([reaction_OtherAsp_R00245])
model_NRRL_35890.reactions.get_by_id('OtherAsp_R00245').bounds = (0,1000)
model_NRRL_35890.reactions.get_by_id('OtherAsp_R00245')

new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id("C00148[e]"), type="sink") # give it extracellular proline
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id("C00014[e]"), type="sink") # give it extracellular NH3
model_NRRL_35890.reactions.get_by_id("r2205").bounds = (0,0) # do not give it glucose
model_NRRL_35890.reactions.get_by_id("OtherAsp_R01248").bounds = (-1000,1000) # make reversible (gibbs = 44), catalyzes L-proline <=> (S)-1-pyrroline-5-carboxylate
model_NRRL_35890.reactions.get_by_id("OtherAsp_R01251").bounds = (-1000,1000) # make reversible (gibbs = 44), catalyzes L-proline <=> (S)-1-pyrroline-5-carboxylate

solution1 = model_NRRL_35890.optimize()
solution1.objective_value
loopless1 = loopless_solution(model_NRRL_35890)
loopless1.objective_value

- L-proline[c] + NAD(+)[c] => hydrogen[c] + NADH[c] + (S)-1-pyrroline-5-carboxylate[c] (cluster_12444)
- L-proline[c] + NADP(+)[c] => hydrogen[c] + NADPH[c] + (S)-1-pyrroline-5-carboxylate[c] (cluster_12444)

- 2 H2O[c] + NAD(+)[c] + (S)-1-pyrroline-5-carboxylate[c] => L-glutamate[c] + hydrogen[c] + NADH[c] (cluster_3022 or cluster_676)
- 2 H2O[c] + NADP(+)[c] + (S)-1-pyrroline-5-carboxylate[c] => L-glutamate[c] + hydrogen[c] + NADPH[c] (cluster_3022 or cluster_676)

In [ ]:
# the path should probably go through glutamate. Would that work?
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id("C00148[e]"), type="sink") # give it extracellular L-glutamate
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id("C00014[e]"), type="sink") # give it extracellular NH3
model_NRRL_35890.reactions.get_by_id("r2205").bounds = (0,0) # do not give it glucose
model_NRRL_35890.objective = {model_NRRL_35890.reactions.get_by_id('r2359'): 1}
solution1 = model_NRRL_35890.optimize()
solution1.objective_value # no

In [ ]:
ensemble.base_model.reactions.get_by_id("OtherAsp_R00708") # OtherAsp_R00708

In [ ]:
model_NRRL_35890.genes.get_by_id("cluster_12444")
model_NRRL_35890.genes.get_by_id("cluster_3022") # neither are present in this isolate. Should be easily gapfilled from pan.
model_NRRL_35890.genes.get_by_id("cluster_676")

In [ ]:
model_NRRL_35890.remove_reactions(["SK_C00025[e]"])
model_NRRL_35890.remove_reactions(["SK_C00014[e]"])

In [ ]:
# Change objective
model_NRRL_35890.objective = {model_NRRL_35890.reactions.get_by_id('SK_C00148[e]'): -1}
solution1 = model_NRRL_35890.optimize()
solution1.objective_value
loopless1 = loopless_solution(model_NRRL_35890)
loopless1.objective_value



## Gapfill pan-oryzae model

On the following conditions:

- L_Proline_PM_01
- L_Arginine_PM_02
- L_Isoleucine_PM_02
- L_Leucine_PM_02
- L_Phenylalanine_PM_02
- L_Valine_PM_02
- Putrescine_PM_02

However, for each of these conditions, there is growth of at least some of the isolates in the plate data. As such, the pan-model should at least be able to grow on it.
In addition, some isolates grow on most of these, e.g., NRRL_35890 grows on all but L_Phenylalanine_PM_02.

In [194]:
# target_mets = ['L_Proline_PM_01', 'L_Arginine_PM_02', 'L_Isoleucine_PM_02', 'L_Leucine_PM_02',
#                'L_Phenylalanine_PM_02', 'L_Valine_PM_02', 'Putrescine_PM_02']

# ! C00077[c]


target_mets = ['C00148[c]', 'C00062[c]', 'C00407[c]', 'C00123[c]', 'C00079[c]', 'C00183[c]', 'C00134[c]', 'C00086[c]']

In [ ]:
ensemble.base_model.medium = medium_Climit
ensemble.base_model.medium

for met in target_mets:
    # Split the string into parts
    # parts = met.split('_')

    # # Take the first part before the second underscore
    # first_part = "_".join(parts[:2])
    # second_part = "_".join(parts[2:])

    # metabolite_value = conditions[(conditions['Medium'] == first_part) & (conditions['plateID'] == second_part)]['Metabolite'].squeeze()
    # metabolite_value = metabolite_value.replace('[e]', '[c]')
    # metabolite_value
    met
    # To allow free access to the metabolite in the cytoplasm, add a sink
    new_sink = ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id(met), type="sink") 

    fluxes = flux_balance.optimize_ensemble(ensemble,
                                            specific_models = submembers,
                                            return_flux = 'r2359') 
    fluxes

    # Remove the sink
    ensemble.base_model.remove_reactions(["SK_" + met])


From this analysis, it is clear that none of the isolates nor the pan-Oryzae model can grow on any of these even if we provide them with the metabolites intracellularly. As such, Id expect to have some missing reactions. Since we also cannot borrow them from the pan-oryzae model, we should try to get them from a universal database.

In [ ]:
# # Import function
# # from cobra.flux_analysis import gapfill

# # Load data
# with open("./scrap/gemList.pickle", 'rb') as infile:
#     gemList = load(infile)

# # Select the target isolate
# # model_NRRL_35890 = next((model for model in gemList if model.id == 'Aspergillus_oryzae_NRRL_2217'), None)
# model_NRRL_35890 = next((model for model in gemList if model.id == 'Aspergillus_oryzae_NRRL_35890'), None)

# # Add metabolite myo-Inositol 4-phosphate to the model
# C03546 = cobra.Metabolite(id="C03546", compartment="c", name="myo-Inositol 4-phosphate")

# # Fix reaction r205
# r205 = model_NRRL_35890.reactions.get_by_id("r205")
# r205.reaction = "C00001[c] + C01220[c] --> C00009[c] + C03546[c]"

# model_NRRL_35890.remove_reactions(reactions = ['OtherAsp_R10479', 'OtherAsp_R10317', 'OtherAsp_R10318'],
#                                      remove_orphans = True)
# model_NRRL_35890.add_reactions([ensemble.base_model.reactions.get_by_id("r766")])
# model_NRRL_35890.reactions.get_by_id("r2205").bounds = (0,1000.0) # in stead of 1,1 or 1,1000, otherwise we get 1,0 -> invalid

# # Store the SOTA NRRL_35890 model
# import pickle
# path = "./scrap/"
# pickle.dump(model_NRRL_35890, open(path+"model_NRRL_35890.pickle","wb"))

In [2]:
import pickle
with open("./scrap/model_NRRL_35890.pickle", 'rb') as infile:
    model_NRRL_35890 = load(infile)

In [ ]:
model_NRRL_35890.medium

In [ ]:
full_medium = deepcopy(model_NRRL_35890.medium)

model_NRRL_35890.medium
model_NRRL_35890.optimize().objective_value # growth, as expected from before

medium_Climit = deepcopy(full_medium)
medium_Climit['r2199'] = 0 # set CO2 to zero
medium_Climit['r2205'] = 0 # set glucose to zero
model_NRRL_35890.medium = medium_Climit
model_NRRL_35890.optimize().objective_value # no carbon source, no growth

In [ ]:
# Give it putrescine
# putrescine C00134, urea C00086, agmatine C00179
# target_mets = ['L_Proline_PM_01', 'L_Arginine_PM_02', 'L_Isoleucine_PM_02', 'L_Leucine_PM_02',
#                'L_Phenylalanine_PM_02', 'L_Valine_PM_02', 'Putrescine_PM_02']
# target_mets = ['C00148[c]', 'C00062[c]', 'C00407[c]', 'C00123[c]', 'C00079[c]', 'C00183[c]', 'C00134[c]', 'C00086[c]']

met = 'C00134[c]'
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")
model_NRRL_35890.optimize().objective_value
model_NRRL_35890.remove_reactions(["SK_" + met])

In [ ]:
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")

# Gapfil from pan-oryzae
panOryzae = next((model for model in gemList if model.id == 'Pan_oryzae'), None)
# result = gapfill(model_NRRL_35890, panOryzae, demand_reactions=False, iterations=4) # INFEASIBLE

In [ ]:
new_sink

In [33]:
# Gapfil from KEGG oryzae
data_dir = Path(".")
data_dir = data_dir.resolve()
model_path = data_dir / "Aor_KEGG.xml"
Aor_KEGG = read_sbml_model(str(model_path.resolve()))
# result = gapfill(model_NRRL_35890, Aor_KEGG, demand_reactions=False, iterations=4) # INFEASIBLE

In [ ]:
# Gapfil from KEGG fumigatus
data_dir = Path(".")
data_dir = data_dir.resolve()
model_path = data_dir / "Afu_KEGG.xml"
Afu_KEGG = read_sbml_model(str(model_path.resolve()))
# result = gapfill(model_NRRL_35890, Afu_KEGG, demand_reactions=False, iterations=4) # INFEASIBLE

In [ ]:
Aor_KEGG.reactions

In [ ]:
# Give it back sugar, optimize flux through putrescine reactions
# particularly OtherAsp_R01151, which breaks it down to some thing that could eventually go to acetylCoa and TCA
model_NRRL_35890.medium = medium_Climit
model_NRRL_35890.medium

# change the objective to ATPM
# model_NRRL_35890.objective = "OtherAsp_R01151"
model_NRRL_35890.objective = "r2359"

# from cobra.util.solver import linear_reaction_coefficients
# linear_reaction_coefficients(model_NRRL_35890)

met = 'C00134[c]'
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")
met = 'C00024[c]'
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")

solution = model_NRRL_35890.optimize()

In [ ]:
# Would it help to have reaction r692 in the cytoplasm?
reaction_r692_c = deepcopy(model_NRRL_35890.reactions.get_by_id("r692"))
reaction_r692_c.reaction = "C00026[c] + C15767[c] --> C00025[c] + C00232[c]"
reaction_r692_c.id = 'r692_cytoplasmic'
model_NRRL_35890.add_reactions([reaction_r692_c])
model_NRRL_35890.reactions.get_by_id('r692_cytoplasmic')

In [ ]:
model_NRRL_35890.reactions.get_by_id('r692_cytoplasmic').bounds = (0,1000)
model_NRRL_35890.reactions.get_by_id('r692_cytoplasmic')

In [ ]:
# If I add r692_cytoplasmic, and give it access to both intracellular putrescine and 2-oxoglutarate, 
# it can grow on medium without glucose!

model_NRRL_35890.medium = medium_Climit
# model_NRRL_35890.medium = full_medium
model_NRRL_35890.medium

# change the objective to ATPM
# model_NRRL_35890.objective = "OtherAsp_R01151"
model_NRRL_35890.objective = "r2359"

# from cobra.util.solver import linear_reaction_coefficients
# linear_reaction_coefficients(model_NRRL_35890)

# met = 'C00134[c]' # putrescine
# new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")
# met = 'C00026[c]' # 2-oxoglutarate
# new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")
met = 'C00077[c]' # L-ornithine
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")
# met = 'C00327[c]' # L-citrulline
# new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")
# met = 'C00062[c]' # L-arginine
# new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")
# met = 'C00025[c]' # L-Glutamate
# new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")
# met = 'C00064[c]' # L-Glutamine
# new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")

solution = model_NRRL_35890.optimize()

In my models, growth on L-Glutamate, L-Glutamine, and L-ornithine (only if including r692 in the cytoplasm) as sole carbon source is supported.
Growth on e.g. putrescine is not supported.

In [ ]:
# 0.1 fix growth rate
reaction_r2359 = model_NRRL_35890.reactions.get_by_id("r2359")
reaction_r2359.bounds = (1, 1) # Set the flux through reaction to a fixed value

# 0.2 add target sinks
met = 'C00077[c]' # L-ornithine
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")
met = 'C00134[c]' # putrescine
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")
met = 'C00026[c]' # 2-oxoglutarate
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")

# 1. allow ornitine uptake and minimize it (have at objective function with -1)
model_NRRL_35890_1 = model_NRRL_35890.copy()
model_NRRL_35890_1.reactions.get_by_id('SK_C00134[c]').bounds = (0,0)
model_NRRL_35890_1.reactions.get_by_id('SK_C00026[c]').bounds = (0,0)
model_NRRL_35890_1.reactions.get_by_id('SK_C00077[c]').bounds = (-1000,1000)

# because a sink is encoded as met<=>, it seems that having a 1 and not a -1 is needed to minimize!
# indeed, if i set it to -1 i get 1000 flux through that reaction
model_NRRL_35890_1.objective = {model_NRRL_35890_1.reactions.get_by_id('SK_C00077[c]'): 1} 
solution1 = model_NRRL_35890_1.optimize()
solution1.objective_value
loopless1 = loopless_solution(model_NRRL_35890_1)

# 2. allow for putrescine and 2-oxo uptake, minimizing 2-oxo uptake
model_NRRL_35890_2 = model_NRRL_35890.copy()
model_NRRL_35890_2.reactions.get_by_id('SK_C00134[c]').bounds = (-1000,1000)
model_NRRL_35890_2.reactions.get_by_id('SK_C00026[c]').bounds = (-1000,1000)
model_NRRL_35890_2.reactions.get_by_id('SK_C00077[c]').bounds = (0,0)
model_NRRL_35890_2.objective = {model_NRRL_35890_2.reactions.get_by_id('SK_C00026[c]'): 1}
solution2 = model_NRRL_35890_2.optimize()
solution2.objective_value
loopless2 = loopless_solution(model_NRRL_35890_2)

# => compare flux distributions

# TODO another question we could ask is: Can putrescine be used to generate ATP?

In [ ]:
rids = [r.id for r in model_NRRL_35890.reactions]

# Calculate the absolute difference for each key
differences = [abs(v1 - v2) for v1, v2 in zip(solution1.fluxes, solution2.fluxes)]

# Sort the keys based on the largest difference
sorted_rids = [rid for _, rid in sorted(zip(differences, rids), reverse=True)]

# Print the sorted keys
print(sorted_rids)

In [ ]:
for sorted_rid in sorted_rids[0:25]:
    print(model_NRRL_35890.reactions.get_by_id(sorted_rid).build_reaction_string(use_metabolite_names = True))

In [ ]:
# perhaps just look at which pathways are on in ornithine and that are off in putrescine?
# Apply the filtering conditions using boolean indexing
filtered_val1 = loopless1.fluxes[(loopless1.fluxes.abs() > 0.5) & (loopless2.fluxes.abs() < 0.1)]
filtered_val1

In [ ]:
# so essentially, with ornithine we have and extra path that goes from threonine to pyruvate
for target in filtered_val1.index:
    print(model_NRRL_35890_1.reactions.get_by_id(target).build_reaction_string(use_metabolite_names = True))

In [ ]:
# citrate <=> H2O + cis-aconitate
# H2O + cis-aconitate <=> isocitrate
# NADP(+) + isocitrate --> NADPH + CO2 + 2-oxoglutarate

# fumarate + FADH2 --> FAD + succinate

# FAD + butyryl-CoA --> trans-but-2-enoyl-CoA + FADH2
# FAD + trans-but-2-enoyl-CoA --> butyryl-CoA + FADH2

# L-ornithine --> CO2 + putrescine

# NAD(+) + glycine + tetrahydrofolate --> NADH + CO2 + NH3 + 5,10-methylenetetrahydrofolate
# glycine + 5,10-methylenetetrahydrofolate <=> L-serine + tetrahydrofolate
# L-serine --> NH3 + pyruvate

# NH3 --> NH3
# L-ornithine <=> L-ornithine
# (S)-malate <=> (S)-malate
# L-glutamate <=> L-glutamate
# ATP + ADP + phosphate --> ATP + ADP + phosphate + H(+)
#  --> NH3
# L-arginine --> 
# L-ornithine <=>

In [ ]:
# so essentially, with ornithine we have and extra path that goes from threonine to pyruvate
filtered_val2 = loopless2.fluxes[(loopless2.fluxes.abs() > 0.1) & (loopless1.fluxes.abs() < 0.001)]
filtered_val2
for target in filtered_val2.index:
    print(model_NRRL_35890.reactions.get_by_id(target).build_reaction_string(use_metabolite_names = True))

In [274]:
# H2O + CO2 --> hydrogen + HCO3
# ATP + pyruvate + HCO3 --> ADP + phosphate + oxaloacetate
# L-glutamate + oxaloacetate <=> 2-oxoglutarate + L-aspartate
# GTP + L-aspartate + IMP --> phosphate + GDP + N6-(1,2-dicarboxyethyl)-AMP
# N6-(1,2-dicarboxyethyl)-AMP <=> AMP + fumarate


# ATP + N-acetyl-L-glutamate --> ADP + N-acetyl-L-gamma-glutamyl phosphate
# NADPH + N-acetyl-L-gamma-glutamyl phosphate --> NADP(+) + phosphate + N-acetyl-L-glutamate 5-semialdehyde
# L-glutamate + N-acetyl-L-glutamate 5-semialdehyde --> 2-oxoglutarate + N2-acetyl-L-ornithine
# L-glutamate + N2-acetyl-L-ornithine --> L-ornithine + N-acetyl-L-glutamate

# ATP + L-glutamate --> ADP + D-alpha-glutamyl phosphate
# NADPH + D-alpha-glutamyl phosphate --> NADP(+) + phosphate + L-glutamate 5-semialdehyde

# NAD(+) + 3-phospho-D-glycerate --> NADH + 3-phosphonooxypyruvate
# L-glutamate + 3-phosphonooxypyruvate --> 2-oxoglutarate + O-phospho-L-serine
# O-phospho-L-serine --> phosphate + L-serine

# NADH + NADP(+) <=> NAD(+) + NADPH
# succinate + (S)-malate <=> succinate + (S)-malate
# L-glutamate + L-aspartate <=> L-glutamate + L-aspartate
# L-asparagine --> 
# L-aspartate --> 
# acetate --> 
# hydrogen --> 
# putrescine <=> 
# 2-oxoglutarate <=> 

In [ ]:
model_NRRL_35890.metabolites.get_by_id('C00624[m]').summary()

In [ ]:
model_NRRL_35890.reactions.get_by_id('r642').build_reaction_string(use_metabolite_names = True)
model_NRRL_35890.reactions.get_by_id('r606').build_reaction_string(use_metabolite_names = True)

In [ ]:
len(solution1.fluxes[solution1.fluxes!=0])
len(solution2.fluxes[solution2.fluxes!=0])
len(loopless1.fluxes[loopless1.fluxes!=0])
len(loopless2.fluxes[loopless2.fluxes!=0])

In [ ]:
model_NRRL_35890.remove_reactions(["SK_C00134[c]"])
model_NRRL_35890.remove_reactions(["SK_C00026[c]"])
model_NRRL_35890.remove_reactions(["SK_C00077[c]"])
model_NRRL_35890.remove_reactions(["SK_C00327[c]"])
model_NRRL_35890.remove_reactions(["SK_C00025[c]"])
model_NRRL_35890.remove_reactions(["SK_C00024[c]"])
model_NRRL_35890.remove_reactions(["SK_C00064[c]"])

In [114]:
# Its ability to grow on ornithine really seems to crucially depend on the ability to generate 2-oxoglutarate
# Most flux goes through r555 and r687
# There is a set of reactions that together only amount to 8% of the total flux. Are all of them necessary?
# Would it die if we delete r2148, OtherAsp_R00734, OtherAsp_R01939, r558, r713, r770, r839, r89, r911

model_NRRL_35890.reactions.get_by_id('r2148').bounds = (0,0) # induces r2178
model_NRRL_35890.reactions.get_by_id('r2178').bounds = (0,0) # can be removed
# model_NRRL_35890.reactions.get_by_id('OtherAsp_R00734').bounds = (-1000,1000) # induces r914 (same reaction) -> cannot be removed 
# model_NRRL_35890.reactions.get_by_id('OtherAsp_R01939').bounds = (-1000,1000) # induces r837 (same reaction) -> cannot be removed
model_NRRL_35890.reactions.get_by_id('r558').bounds = (0,0)  # can be removed
model_NRRL_35890.reactions.get_by_id('r713').bounds = (0,0)  # can be removed
# model_NRRL_35890.reactions.get_by_id('r770').bounds = (0,1000)  # cannot be removed
model_NRRL_35890.reactions.get_by_id('r839').bounds = (-1000,1000) # cannot be removed
model_NRRL_35890.reactions.get_by_id('r89').bounds = (0,0) # can be removed
model_NRRL_35890.reactions.get_by_id('r911').bounds = (-1000,1000) # induces OtherAsp_R00694 (same reaction) -> cannot be removed

In [ ]:
solution = model_NRRL_35890.optimize()
solution.objective_value

In [ ]:
model_NRRL_35890.metabolites.get_by_id("C00036[c]")
model_NRRL_35890.metabolites.get_by_id("C00036[c]").summary()

model_NRRL_35890.metabolites.get_by_id("C00033[c]")
model_NRRL_35890.metabolites.get_by_id("C00033[c]").summary()

In [ ]:
# model_NRRL_35890.reactions.get_by_id("r645").summary()
# model_NRRL_35890.reactions.get_by_id("r2125").summary()
model_NRRL_35890.reactions.get_by_id("r692_cytoplasmic").summary()

In [ ]:
# model_NRRL_35890.reactions.get_by_id("r692_cytoplasmic")
model_NRRL_35890.reactions.get_by_id("r104")
model_NRRL_35890.reactions.get_by_id("r104").summary()

In [8]:
from d3flux import flux_map

# Create a new model with the selected reactions
subset_model = model_NRRL_35890.copy()

# Filter the reactions based on the given identifiers
subset_model.reactions = [r for r in model_NRRL_35890.reactions if r.id in ["r2125", "r645", "r643", "r606", "r2171", "r607", "r608", 
                                                                            "r693", "r1109", "r101", "r2155", "r104", "r555", "OtherAsp_R01151",
                                                                            "r964", "r692_cytoplasmic", "r676", "r2156", "r97", "r100"]]

used_metabolites = set()
for reaction in subset_model.reactions:
    used_metabolites.update(reaction.metabolites.keys())
id_list = [obj.id for obj in used_metabolites]

# Only retain metabolites that are used in the model
subset_model.metabolites = [m for m in subset_model.metabolites if m.id in id_list]

flux_map(subset_model, display_name_format=lambda x: str(x.id), figsize=(850,850),
         flux_dict={rxn.id: None for rxn in subset_model.reactions})

In [ ]:
# How could it be possible that it grows on L-ornithine and not on putrescine?
model_NRRL_35890.reactions.get_by_id("r2125").summary() # transport ornithin to mitochondria (1)
model_NRRL_35890.reactions.get_by_id("r645").summary() # make CO2 and putrescine (2)
model_NRRL_35890.reactions.get_by_id("r643").summary() # make 2-oxoglutarate[c] + L-ornithine[c] => L-glutamate[c] + L-glutamate 5-semialdehyde[c] (3)

################################# 1
model_NRRL_35890.reactions.get_by_id("r606").summary() # carbamoyl phosphate[m] + L-ornithine[m] => L-citrulline[m] + phosphate[m] 
# where does the carbamoyl phosphate[m] come from. It comes from the cytoplasm, but how? (1.1)

model_NRRL_35890.reactions.get_by_id("r2171").summary() # L-citrulline back to cytoplasm
model_NRRL_35890.reactions.get_by_id("r607").summary() # L-aspartate[c] + ATP[c] + L-citrulline[c] <=> AMP[c] + N-(L-arginino)succinate[c] + diphosphate[c]
# Where does the aspartate come from? (1.2)

model_NRRL_35890.reactions.get_by_id("r608").summary() # N-(L-arginino)succinate[c] <=> L-arginine[c] + fumarate[c]
# argine: make a bit of protein, but throw 99% out
# fumarate: put it into mitochondria -> make (S)-malate[m]

# (1.1) where does the carbamoyl phosphate[c] come from
# From (3), glutamate is produced.
model_NRRL_35890.reactions.get_by_id("r693").summary() # ATP[c] + L-glutamate[c] + NH3[c] => ADP[c] + L-glutamine[c] + phosphate[c]
model_NRRL_35890.reactions.get_by_id("r1109").summary()  # 2 ATP[c] + CO2[c] + L-glutamine[c] => 2 ADP[c] + carbamoyl phosphate[c] + L-glutamate[c] + phosphate[c]

# (1.2) where does the L-aspartate come from?
# From S-malate? But it cant come from there without L-aspartate???
# -> it can through putrescine! See below in (2)
model_NRRL_35890.reactions.get_by_id("r101").summary() # (S)-malate[m] + NAD(+)[m] <=> NADH[m] + oxaloacetate[m]
model_NRRL_35890.reactions.get_by_id("r2155").summary() # oxaloacetate[m] <=> oxaloacetate[c] (mostly)
model_NRRL_35890.reactions.get_by_id("r104").summary() # ATP[c] + citrate[c] + coenzyme A[c] => acetyl-CoA[c] + ADP[c] + oxaloacetate[c] + phosphate[c]
model_NRRL_35890.reactions.get_by_id("r555").summary() # L-glutamate[c] + oxaloacetate[c] <=> 2-oxoglutarate[c] + L-aspartate[c]

################################# 2
# From putrescine, we continue:
# OtherAsp_R01151: H2O + oxygen + putrescine --> NH3 + hydrogen peroxide + 4-aminobutanal
# r964: NAD(+) + 4-aminobutanal --> NADH + gamma-aminobutyrate
# r692_cytoplasmic: 2-oxoglutarate + gamma-aminobutyrate --> L-glutamate + succinate semialdehyde (note: adding this reaction from m to c was essential) But where does the 2-oxoglutarate come from
# r676: H2O[c] + NAD(+)[c] + succinate semialdehyde[c] => NADH[c] + succinate[c]
# r2156: succinate[c] => succinate[m]
# r97: ubiquinone[m] + succinate[m] <=> fumarate[m] + ubiquinol[m]
# r100: fumarate[m] + H2O[m] <=> (S)-malate[m]


################################# 3 
# The big question remains, where does the 2-oxoglutarate[c] necessary for (3) come from? And it is also necessary in (2) for r692_cytoplasmic.

## it is used in three reactions
# r685:  2-oxoglutarate[c] + NADH[c] + NH3[c] <=> L-glutamate[c] + NAD(+)[c]
# r692_cytoplasmic: 2-oxoglutarate + gamma-aminobutyrate => L-glutamate + succinate semialdehyde
# r643: 2-oxoglutarate[c] + L-ornithine[c] => L-glutamate[c] + L-glutamate 5-semialdehyde[c]

## it is generated in reactions:
# 70% in r555: L-glutamate[c] + oxaloacetate[c] => 2-oxoglutarate[c] + L-aspartate[c]
# 27% in r687: L-glutamate[c] + H2O[c] + NADP(+)[c] => 2-oxoglutarate[c] + NADPH[c] + NH3[c]

# 0.26% in OtherAsp_R00734: 3-(4-hydroxyphenyl)pyruvate[c] + L-glutamate[c] => 2-oxoglutarate[c] + L-tyrosine[c]
# 0.53% in OtherAsp_R01939: 2-oxoadipate[c] + L-glutamate[c] => 2-oxoglutarate[c] + L-2-Aminoadipate[c]
# 0.19%	in r770: L-glutamate[c] + 3-(imidazol-4-yl)-2-oxopropyl phosphate[c] => 2-oxoglutarate[c] + L-histidinol phosphate[c]
# 0.53% in r839: NAD(+)[c] + L-saccharopine[c] => 2-oxoglutarate[c] + L-lysine[c] + NADH[c]
# 0.29% in r911: L-glutamate[c] + keto-phenylpyruvate[c] => 2-oxoglutarate[c] + L-phenylalanine[c]

# So basically, it is always used to make Glutamate (in all 3 reactions), and all reactions to make it require Glutamate, except for r839????
# But this also needs glutamate
# L-2-aminoadipate 6-semialdehyde[c] + L-glutamate[c] + NADPH[c] <=> L-saccharopine[c] + NADP(+)[c]

# So then this should just not be possible, right? If all reactions that use 2-oxoglutarate to make Glutamate require Glutamate, then there can be no Glutamate...
# To close my week, I just jot down all reactions that have flux for making Glutamate:


# 0.26% in r1030: ATP[c] + N(2)-formyl-N(1)-(5-phospho-D-ribosyl)glycinamide[c] + L-glutamine[c] => ADP[c] + 2-formamido-N(1)-(5-phospho-D-ribosyl)acetamidine[c] + L-glutamate[c] + phosphate[c]
# 0.26% in r1033: L-glutamine[c] + 5-phospho-alpha-D-ribose 1-diphosphate[c] => L-glutamate[c] + diphosphate[c] + 5-phospho-D-ribosylamine[c]
# 0.10% in r1040: ATP[c] + L-glutamine[c] + xanthosine 5'-phosphate[c] => AMP[c] + L-glutamate[c] + GMP[c] + diphosphate[c]
# 37.13% in r1109: 2 ATP[c] + CO2[c] + L-glutamine[c] => 2 ADP[c] + carbamoyl phosphate[c] + L-glutamate[c] + phosphate[c]
# 0.90% in r2162: L-glutamate[c] <=> L-glutamate[m]
# 0.26% in r643: 2-oxoglutarate[c] + L-ornithine[c] => L-glutamate[c] + L-glutamate 5-semialdehyde[c]
# 40.04% in r685: 2-oxoglutarate[c] + NADH[c] + NH3[c] <=> L-glutamate[c] + NAD(+)[c]
# 19.85% in r692_cytoplasmic: 2-oxoglutarate + gamma-aminobutyrate => L-glutamate + succinate semialdehyde
# 0.66% in r701: beta-D-fructofuranose 6-phosphate[c] + L-glutamine[c] => D-glucosamine 6-phosphate[c] + L-glutamate[c]
# 0.11% in r763: L-glutamine[c] + 5-[(5-phospho-1-deoxy-D-ribulos-1-ylamino)methylideneamino]-1-(5-phospho-D-ribosyl)imidazole-4-carboxamide[c] => 5-amino-1-(5-phospho-D-ribosyl)imidazole-4-carboxamide[c] + D-erythro-1-(imidazol-4-yl)glycerol 3-phosphate[c] + L-glutamate[c]
# 0.42% in r885: chorismate[c] + L-glutamine[c] => anthranilate[c] + L-glutamate[c] + pyruvate[c]

# So, all relying on either L-glutamine or 2-oxoglutarate, or L-glutamate[m]

# !! There is only 1 reaction producing Glutamine: r693
# r693: ATP[c] + L-glutamate[c] + NH3[c] => ADP[c] + L-glutamine[c] + phosphate[c]
# And it requires Glutamate -_- pain

# There is only 1 reaction producing L-glutamate[m]
# r683:  2-oxoglutarate[m] + NADH[m] + NH3[m] => L-glutamate[m] + NAD(+)[m]
# With

# 13.31% r781: 2-oxoglutarate[m] + L-isoleucine[m] <=> L-glutamate[m] + 2-keto-3-methyl-valerate[m]
# 18.93% r782: 2-oxoglutarate[m] + L-valine[m] <=> L-glutamate[m] + (R)-3-methyl-2-oxobutanoate[m]
# 20.41% r783: 2-oxoglutarate[m] + L-leucine[m] <=> L-glutamate[m] + (2S)-2-isopropyl-3-oxosuccinate[m]
# 47.34% r87: isocitrate[m] + NAD(+)[m] => 2-oxoglutarate[m] + CO2[m] + NADH[m]

# and cis-aconitate[m] + H2O[m] <=> isocitrate[m]
# and citrate[m] <=> cis-aconitate[m] + H2O[m]
# and citrate[c] + (S)-malate[m] <=> (S)-malate[c] + citrate[m]
# and citrate[c] <=> acetate[c] + oxaloacetate[c]
# And then acetate[c] and oxaloacetate[c] come from several reactions.......

In [ ]:
model_NRRL_35890.reactions.get_by_id("r608").summary() 

In [ ]:
model_NRRL_35890.reactions.get_by_id("OtherAsp_R00111").summary()
model_NRRL_35890.reactions.get_by_id("r607").summary()
model_NRRL_35890.reactions.get_by_id("OtherAsp_R00557").summary()

In [ ]:
model_NRRL_35890.metabolites.get_by_id("C03406[c]")

In [ ]:
model_NRRL_35890.remove_reactions(["SK_C00134[c]"])
model_NRRL_35890.remove_reactions(["SK_C00026[c]"])
model_NRRL_35890.remove_reactions(["SK_C00077[c]"])
model_NRRL_35890.remove_reactions(["SK_C00327[c]"])

In [ ]:
met = 'C00134[c]'
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id(met), type="sink")
result = gapfill(model_NRRL_35890, Aor_KEGG, demand_reactions=False, iterations=4)

In [ ]:
# First, take the NRRL_35890 model and assess growth on the different carbon sources after putting them into the cytoplasm
# Perhaps we can maximize for something other than growth and see which reactions carry high flux to identify cycles?
from cobra.flux_analysis import gapfill
with open("./scrap/gemList.pickle", 'rb') as infile:
    gemList = load(infile)
model_NRRL_35890 = next((model for model in gemList if model.id == 'Aspergillus_oryzae_NRRL_35890'), None)
model_NRRL_35890.medium = medium_Climit
new_sink = model_NRRL_35890.add_boundary(model_NRRL_35890.metabolites.get_by_id('C00062[c]'), type="sink")
model_NRRL_35890.optimize().objective_value # no growth, as expected from before


# Then, see if we can get growth by gapfilling from the Pan-oryzae model. Perhaps the required reaction(s) is there, but other reactions are avoiding growth.
data_dir = Path(".")
data_dir = data_dir.resolve()
model_path = data_dir / "panAsp_v2.xml"
panOryzae = read_sbml_model(str(model_path.resolve()))

# Line below fails, not possible to get growth by adding reactions from the the Pan-oryzae model!
# result = gapfill(model_NRRL_35890, panOryzae, demand_reactions=False, iterations=4)

# Then, see if we can get growth by gapfilling from the KEGG oryzae model.
data_dir = Path(".")
data_dir = data_dir.resolve()
model_path = data_dir / "Aor_KEGG.xml"
Aor_KEGG = read_sbml_model(str(model_path.resolve()))
# Then, last resort, see if we can get growth by gapfilling from the universal KEGG database (any organism).

In [ ]:
# Fails as well
result = gapfill(model_NRRL_35890, Aor_KEGG, demand_reactions=False, iterations=4)


In [ ]:
# I want growth on L-arginine ('C00062[c]').capitalize
data_dir = Path(".")
data_dir = data_dir.resolve()
model_path = data_dir / "completeKEGG.xml"
universal_KEGG = read_sbml_model(str(model_path.resolve()))

In [ ]:
# Allow internalization of each of these metabolites:
import numpy as np
np.random.seed(333)

growth = []
oxygen = []
ensemble.base_model.medium = deepcopy(medium_Climit)

for i in range(len(conditions.index)):

    # Generate the corresponding metabolite sink
    new_sink = ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id(conditions['Metabolite'].iloc[i]), type="sink") 

    # Only storing two evetually for now, but these are fluxes where I peviously saw differences across members
    fluxes = flux_balance.optimize_ensemble(ensemble,
                                            specific_models = submembers,
                                            return_flux = ['r2359', 'r2199']) 
    growth.append(fluxes['r2359'].values)
    oxygen.append(fluxes['r2202'].values)

    # remove metabolite sink
    ensemble.base_model.remove_reactions(["SK_" + conditions['Metabolite'].iloc[i]])

## Growth on agmatine, urea, and/or putrescine

1. agmatine: even if we give intracellular agmatine, no growth!
2. putrescine: even if we give intracellular putrescine, no growth!
3. urea: having a urea transporter would work! It would mean it could grow on urea[e] and thus also agmatine[e] if gene cluster_9575 is present, which however is only true for Aspergillus_oryzae_NRRL_5589	

In [ ]:
# putrescine C00134, urea C00086, agmatine C00179, L-arginine C00062, L-ornithine C00077
# Id expect growth on any of these to go through the same pathway.

target = ["C00134[c]", "C00086[c]", "C00179[c]", "C00062[c]", "C00077[c]"]

for i in range(len(target)):
    target[i]
    ensemble.base_model.medium = medium_Nlimit.copy()

    # Try if it grows with intracellular sink
    new_sink = ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id(target[i]), type="sink")
    fluxes = flux_balance.optimize_ensemble(ensemble,
                                            specific_models = submembers,
                                            return_flux = ['r2359', 'r2199', 'r2200', 'r2202', 'r2203',
                                                           'r2204', 'r2205', 'r2298', 'r2317', 'r1901',
                                                           'r2333', 'r2339', 'r2357', 'r2360'])
    fluxes
    ensemble.base_model.remove_reactions([new_sink.id])


In [ ]:
ensemble.base_model.genes.get_by_id("cluster_9575")
ensemble.base_model.genes.get_by_id("cluster_9560")
ensemble.base_model.genes.get_by_id("cluster_3072")
ensemble.base_model.genes.get_by_id("cluster_6502")
ensemble.base_model.genes.get_by_id("cluster_11539")

In [ ]:
ensemble.base_model.remove_reactions("DM_C00315[c]")

In [ ]:
# Allow out:
# ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id("C00750[c]"), type="demand") # spermin
ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id("C00170[c]"), type="demand") # 5'-S-methyl-5'-thioadenosine
ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id("C00315[c]"), type="demand") # spermidin 

# Give it:
ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id("C00179[c]"), type="sink") # agmatin
# ensemble.base_model.add_boundary(ensemble.base_model.metabolites.get_by_id("C01137[c]"), type="sink") # S-adenosylmethioninamine

In [ ]:
ensemble.base_model.medium = deepcopy(medium_Nlimit)
ensemble.base_model.medium

In [ ]:
# # change the objective to r612
# ensemble.base_model.objective = "r612" # or r612

# # The upper bound should be 1000, so that we get
# # the actual optimal value
# ensemble.base_model.reactions.get_by_id("r612").upper_bound = 1000
# from cobra.util.solver import linear_reaction_coefficients
# linear_reaction_coefficients(ensemble.base_model)

# #ensemble.base_model.optimize().objective_value

In [ ]:
fluxes = flux_balance.optimize_ensemble(ensemble,
                                        specific_models = submembers,
                                        return_flux = ['r2359', 'r2202', 'r2205',
                                                       'SK_C00179[c]', 'r612',
                                                       'r593', 'r594', 'r595', 'r938', 'r939']) 
fluxes

In [ ]:
ensemble.base_model.remove_reactions("DM_C00170[c]")

# Isolate NRRL_35890

Most mistakes in growth prediction are all in one particular isolate, NRRL_35890. Probably, our model is missing a crucial gene for this isolate, that should in reality be present. How can we find this gene usign cobrapy and medusa?

In [ ]:
ensemble.members.Aspergillus_oryzae_NRRL_35890.states

In [ ]:
# This is the only gene I could fine that is absent from the Aspergillus_oryzae_NRRL_35890 model,
# but present in all 8 other isolates. Not sure if this is uesful; maybe Aspergillus_oryzae_NRRL_35890
# doesnt grow because of the presence of a gene, the combined loss of multiple genes, ...
# Are there functions for assessing this?

ensemble.base_model.genes.get_by_id("cluster_12484")
ensemble.base_model.reactions.get_by_id("r766")

# Biomass specification

In yeast8.7.1, we have:
- 55.3 ATP[c] + 55.3 H2O[c] + lipid[c] + protein[c] + carbohydrate[c] + RNA[c] + DNA[c] + cofactor[c] + ion[c] => 55.3 ADP[c] + biomass[c] + 55.3 H+[c] + 55.3 phosphate[c]

- 0.52701 Ala-tRNA(Ala)[c] + 0.18459 Arg-tRNA(Arg)[c] + 0.11682 Asn-tRNA(Asn)[c] + 0.34173 Asp-tRNA(Asp)[c] + 0.0075813 Cys-tRNA(Cys)[c] + 0.12107 Gln-tRNA(Gln)[c] + 0.34667 Glu-tRNA(Glu)[c] + 0.33358 Gly-tRNA(Gly)[c] + 0.076157 His-tRNA(His)[c] + 0.22135 Ile-tRNA(Ile)[c] + 0.34047 Leu-tRNA(Leu)[c] + 0.32875 Lys-tRNA(Lys)[c] + 0.058238 Met-tRNA(Met)[c] + 0.15381 Phe-tRNA(Phe)[c] + 0.18919 Pro-tRNA(Pro)[c] + 0.21296 Ser-tRNA(Ser)[c] + 0.21986 Thr-tRNA(Thr)[c] + 0.032622 Trp-tRNA(Trp)[c] + 0.11716 Tyr-tRNA(Tyr)[c] + 0.30394 Val-tRNA(Val)[c] => 0.52701 tRNA(Ala)[c] + 0.18459 tRNA(Arg)[c] + 0.11682 tRNA(Asn)[c] + 0.34173 tRNA(Asp)[c] + 0.0075813 tRNA(Cys)[c] + 0.12107 tRNA(Gln)[c] + 0.34667 tRNA(Glu)[c] + 0.33358 tRNA(Gly)[c] + 0.076157 tRNA(His)[c] + 0.22135 tRNA(Ile)[c] + 0.34047 tRNA(Leu)[c] + 0.32875 tRNA(Lys)[c] + 0.058238 tRNA(Met)[c] + 0.15381 tRNA(Phe)[c] + 0.18919 tRNA(Pro)[c] + 0.21296 tRNA(Ser)[c] + 0.21986 tRNA(Thr)[c] + 0.032622 tRNA(Trp)[c] + 0.11716 tRNA(Tyr)[c] + 0.30394 tRNA(Val)[c] + protein[c]

- 0.044535 AMP[c] + 0.043276 CMP[c] + 0.044535 GMP[c] + 0.057992 UMP[c] => RNA[c]

- 0.0036 dAMP[c] + 0.0024 dCMP[c] + 0.0024 dGMP[c] + 0.0036 dTMP[c] => DNA[c]

- lipid backbone[c] + lipid chain[c] => lipid[c]
- 0.0069103 1-phosphatidyl-1D-myo-inositol backbone[c] + 0.026583 ergosterol[c] + 0.0068058 ergosterol ester backbone[c] + 0.0014801 fatty acid backbone[c] + 0.0059508 phosphatidyl-L-serine backbone[c] + 0.025783 phosphatidylcholine backbone[c] + 0.0069293 phosphatidylethanolamine backbone[c] + 0.0068571 triglyceride backbone[c] => lipid backbone[c]
- 0.0080858 C16:0 chain[c] + 0.02373 C16:1 chain[c] + 0.0022663 C18:0 chain[c] + 0.0087066 C18:1 chain[c] => lipid chain[c]

- 0.74851 (1->3)-beta-D-glucan[ce] + 0.25009 (1->6)-beta-D-glucan[ce] + 0.36141 glycogen[c] + 0.71094 mannan[c] + 0.13828 trehalose[c] => carbohydrate[c]

- 0.00019 coenzyme A[c] + 1e-05 FAD[c] + 0.00265 NAD[c] + 0.00015 NADH[c] + 0.00057 NADP(+)[c] + 0.0027 NADPH[c] + 0.00099 riboflavin[c] + 1.2e-06 TDP[c] + 6.34e-05 THF[c] + 1e-06 heme a[c] => cofactor[c]

- 3.04e-05 iron(2+)[c] + 0.00363 potassium[c] + 0.00397 sodium[c] + 0.02 sulphate[c] + 0.00129 chloride[c] + 0.00273 Mn(2+)[c] + 0.000748 Zn(2+)[c] + 0.000217 Ca(2+)[c] + 0.0012425 Mg(2+)[c] + 0.000659 Cu2(+)[c] => ion[c]


In Aspergillus oryzae, we currently have:
- 1.5145 1,3-beta-D-glucan[c] + 49 ATP[c] + 0.40759 chitin[c] + 0.02836 deoxyribonucleic acids[c] + 0.01365 free fatty acids[c] + 0.08952 glycerol[c] + 0.00212 glycogen[c] + 0.21333 D-mannitol[c] + 0.00903 phosphatidylamine[c] + 0.03356 phosphatidylcholines[c] + 0.01468 phosphatidylethanolamines[c] + 3.5008 protein[c] + 0.00564 phosphatidylserine[c] + 0.18259 ribonucleic acids[c] + 0.02617 triglycerides[c] => 49 ADP[c] + biomass[c] + 49 phosphate[c]

- 1.6755 1,3-beta-D-glucan[c] + 49 ATP[c] + 0.45091 chitin[c] + 0.02833 deoxyribonucleic acids[c] + 0.01163 free fatty acids[c] + 0.00212 glycogen[c] + 0.35514 D-mannitol[c] + 0.00769 phosphatidylamine[c] + 0.02859 phosphatidylcholines[c] + 0.0125 phosphatidylethanolamines[c] + 3.2344 protein for biomass and amylase[c] + 0.00481 phosphatidylserine[c] + 0.16861 ribonucleic acids[c] + 0.0223 triglycerides[c] => 49 ADP[c] + biomass[c] + 49 phosphate[c]


Note that these two definitions of the biomass function, called "Biomass formation" and "Biomass formation CF", respectively, only differ in the fact that the second definition does not include Glycerol (and some minor corresponding changes in the coefficients).

So, in principle, we could add "+ cofactor[c]" to the right-hand side of the biomass function, and then specify the formation of cofactor[c] in an additional reaction, where we take only the cofactors already described in our Aspergillus GEM (NAD, NADH, NADP, NADPH, FAD, FADH2, coenzyme A, riboflavin -is this a cofactor?-, heme a; we do not have TDP and THF as such, we do have some derivatives). But then, what would the coefficients need to be?

